# Giai đoạn 17: Validation ngoài trên ScaleOLDNEW (theo protocol v1.6 đã khoá)

Notebook này làm **đúng và chỉ** những gì protocol `paper/external_validation_protocol.md` (v1.6, đã ký) quy định. Không thử thêm mô hình, cấu hình, tham chiếu hay cách cắt cửa sổ nào khác.

| Phần | Nội dung | Mục protocol | Runtime |
|---|---|---|---|
| 17A | Train 5 encoder cuối trên 15 bệnh nhân ds004752 (13 để tối ưu + 2 để early stopping, không refit) | 3 | GPU, ~1–2 giờ |
| 17B | Dựng dữ liệu ScaleOLDNEW: quy tắc kênh, target, cửa sổ, chia calib/test, 3 mức đánh giá + **execution log** | 4, 5, 6, 9 | CPU/GPU, ~30–60 phút (đọc lại 62 GB raw để đếm giá trị non-finite và tìm khoảng mất tín hiệu) |
| ⏸ | **Dừng: gửi output 17B cho mình kiểm tra** trước khi chạy bất kỳ mô hình nào | 9 | |
| 17C | Readout ở mức chính: Final + Base, Learned-3, Fixed-3, Fixed-1, Final với calib 10% | 3, 7 | GPU, ~2–3 giờ |
| 17D | H1, H2 ở mức phụ (artefact-screened) và mức sensitivity | 6, 7 | GPU, ~2 giờ |
| 17E | Phân tích common mode theo cấu hình ampli | 8 | CPU, ~15 phút |
| 17F | Báo cáo: H1/H2 (Wilcoxon hai phía, Holm), các phân tích mô tả | 7 | vài giây |

Mọi bước **tiếp tục được khi bị ngắt** (phần đã xong được bỏ qua).

**RAM:** ScaleOLDNEW có 30–40 kênh scalp, gần gấp đôi ds004752, nên ma trận thống kê của mô hình cuối lớn khoảng 0,5 GB mỗi cái. Nếu 17C/17D bị crash vì hết RAM, chuyển sang runtime **High-RAM** rồi chạy lại cell đó.

In [ ]:
#@title 1. Mount Drive + đường dẫn + cài đặt
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
DATA_DIR = f'{ROOT}/data/s2i_v1'          # dữ liệu train encoder của bài (giống hệt các encoder đã báo cáo)
SC_ROOT  = f'{ROOT}/data/scalesoldnew'    # output Phase 14 (proc/, raw/)
DATA17   = f'{ROOT}/data/scales_s2i'      # output 17B: primary/, secondary/, sensitivity/
ENC17    = f'{ROOT}/runs/p17_encoders'    # 5 encoder cuối
RUNS17   = f'{ROOT}/runs/p17'             # readout ScaleOLDNEW
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
!pip -q install mne==1.13.2 openpyxl
print('ds004752:', len(glob.glob(f'{DATA_DIR}/sub-[0-9][0-9].pt')), 'bệnh nhân (cần 15)')
print('ScaleOLDNEW: proc', len(glob.glob(f'{SC_ROOT}/proc/*.npz')), '/ 84 block, raw .dat',
      len(glob.glob(f'{SC_ROOT}/raw/SEEG*/Block*/*.dat')), '/ 84, bảng kênh', len(glob.glob(f'{SC_ROOT}/raw/SEEG*/*ChannelData.xlsx')), '/ 12')

## 2. Ghi code lên Drive

In [ ]:
%%writefile {CODE_DIR}/s2i/shared.py
"""
Phase 2 - shared temporal encoder + per-subject ridge decoder (LOSO, GPU).

Evidence from phases 0-1d (15 subjects, bipolar iEEG, CAR EEG, 20 % chronological calibration):
  * the SPATIAL scalp -> contact mapping does not transfer across subjects (1a: zero-shot r ~ 0);
  * within a subject the lever is frequency-selective, long-support LINEAR temporal filtering
    (1d: random Morlet filter bank, linear, gated on top of the ridge: +0.0075, 14/15 subjects),
    while non-linearity and more random features add ~nothing at 4 min of calibration.
Hypothesis: the TEMPORAL part (which filters of the scalp signal predict intracranial activity)
is shared across subjects even though the spatial part is not. Hence:
  encoder : per-channel filter bank, shared by all channels and all subjects, trained on the
            OTHER subjects. Quadrature pairs initialised as complex Morlet 2-60 Hz (+-1.2 s
            support) + the envelope of each pair + a few free filters.
  decoder : per-subject linear readout (channels x filters -> contacts), fitted in closed form by
            ridge on the calibration trials only, exactly like the baselines.
Training is episodic so it matches deployment: each step takes a training subject, fits a ridge
readout on ~4 min of consecutive trials (support), scores Pearson r on the following trials
(query), and back-propagates through the ridge solve into the filters. Model selection: the
same procedure on 2 held-out validation subjects (first 20 % of trials -> later trials).

Evaluation on the test subject (never seen in training; same split and calibration sizes as
phases 0-1d, so ridge / base / lin_gated must reproduce the phase-1d numbers):
  ridge, base, lin_gated  baselines (lin_gated = best model of phase 1d: the bar to beat)
  enc0_gated              base + gated stage 2 on the encoder AT INITIALISATION (untrained Morlet
                          bank): control - does the gain come from learning or from the init?
  lin_ridge, enc0_ridge   single-stage controls: ridge (K-fold, shared alpha) on the random linear
                          Morlet bank / on the UNTRAINED encoder
  enc_ridge               ridge (K-fold, shared alpha) on learned encoder features alone
  enc_ridge_cal           enc_ridge with a per-contact amplitude gain fitted on out-of-fold
                          calibration predictions (same r, honest R^2)
Every model is also scored with
  r_no50     46-54 Hz removed from prediction and target (the trained filters develop a sharp 50 Hz
             peak: mains residue after the notch filter);
  r_induced  each side minus its own trial average over the test trials, i.e. how well
             the SINGLE-TRIAL (non stimulus-locked) part is predicted.
  enc_gated               base + gated stage 2 on learned encoder features      <- main model

Usage
-----
python -m s2i.shared --data DATA_DIR --out RUNS_DIR/shared_bip_car
"""
import argparse
import copy
import glob
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from scipy.signal import butter, sosfiltfilt

from s2i.ceiling import StreamRidge, chrono_split, load_trials, template, tmpl_at
from s2i.metrics import per_contact_metrics
from s2i.nonlin import RandomConvFeatures, gated, summarize
from s2i.within import ALPHAS_WIDE, LAG9, KFoldRidge

FS = 200
EXTRA_METRICS = ("r_no50", "r_induced")


# ----------------------------------------------------------------------------- encoder
class TemporalEncoder(nn.Module):
    """Per-channel filter bank shared by all channels and subjects: (B, C, L) -> (B, C, F, L)."""

    def __init__(self, n_pairs=12, half=240, fmin=2.0, fmax=60.0, cycles=5.0, n_free=4, envelope=True):
        super().__init__()
        self.cfg = dict(n_pairs=n_pairs, half=half, fmin=fmin, fmax=fmax, cycles=cycles, n_free=n_free,
                        envelope=envelope)
        self.n_pairs, self.envelope = n_pairs, envelope
        t = np.arange(-half, half + 1) / FS
        f = np.geomspace(fmin, fmax, n_pairs)
        gauss = np.exp(-t[None] ** 2 / (2 * (cycles / (2 * np.pi * f[:, None])) ** 2))
        w = np.zeros((2 * n_pairs + n_free, t.size))
        w[0:2 * n_pairs:2] = gauss * np.cos(2 * np.pi * f[:, None] * t[None])
        w[1:2 * n_pairs:2] = gauss * np.sin(2 * np.pi * f[:, None] * t[None])
        if n_free:
            w[2 * n_pairs, half] = 1.0                                   # identity (broadband, lag 0)
            g = np.random.default_rng(0)
            w[2 * n_pairs + 1:] = g.normal(size=(n_free - 1, t.size)) * np.exp(-t ** 2 / (2 * 0.05 ** 2))
        w /= np.linalg.norm(w, axis=1, keepdims=True)
        self.conv = nn.Conv1d(1, w.shape[0], 2 * half + 1, padding=half, bias=False)
        with torch.no_grad():
            self.conv.weight.copy_(torch.tensor(w, dtype=torch.float32)[:, None])

    @property
    def n_feat(self):
        return self.conv.out_channels + (self.n_pairs if self.envelope else 0)

    def forward(self, x):
        B, C, L = x.shape
        h = self.conv(x.reshape(B * C, 1, L))
        if self.envelope:
            p = h[:, :2 * self.n_pairs].float()
            h = torch.cat([h.float(), torch.sqrt(p[:, 0::2] ** 2 + p[:, 1::2] ** 2 + 1e-6)], 1)
        return h.reshape(B, C, -1, L)


# ----------------------------------------------------------------------------- torch helpers
def pad_trials(E, Y, idx, dev):
    L = max(E[i].shape[1] for i in idx)
    x = np.zeros((len(idx), E[idx[0]].shape[0], L), np.float32)
    y = np.zeros((len(idx), Y[idx[0]].shape[0], L), np.float32)
    m = np.zeros((len(idx), L), bool)
    for j, i in enumerate(idx):
        n = E[i].shape[1]
        x[j, :, :n], y[j, :, :n], m[j, :n] = E[i], Y[i], True
    return torch.from_numpy(x).to(dev), torch.from_numpy(y).to(dev), torch.from_numpy(m).to(dev)


def rows(enc, E, Y, idx, dev, amp, chunk=16):
    """Encoder features and targets of the given trials as (N, C*F) / (N, K) rows (padding removed)."""
    X, T = [], []
    for s in range(0, len(idx), chunk):
        x, y, m = pad_trials(E, Y, idx[s:s + chunk], dev)
        with torch.autocast(device_type=dev.type, enabled=amp):
            h = enc(x)
        B, C, F, L = h.shape
        X.append(h.float().permute(0, 3, 1, 2).reshape(B, L, C * F)[m])
        T.append(y.permute(0, 2, 1)[m])
    return torch.cat(X), torch.cat(T)


def ridge_torch(Xs, Ys, Xq, lam):
    """Ridge on standardised features; lam is relative to the mean eigenvalue (= 1 after scaling)."""
    mu, sd = Xs.mean(0), Xs.std(0) + 1e-4
    A, Q = (Xs - mu) / sd, (Xq - mu) / sd
    ym = Ys.mean(0)
    N, D = A.shape
    G = A.T @ A / N + lam * torch.eye(D, device=A.device)
    return Q @ torch.linalg.solve(G, A.T @ (Ys - ym) / N) + ym


def col_r(P, T):
    P, T = P - P.mean(0), T - T.mean(0)
    return (P * T).sum(0) / ((P * P).sum(0).sqrt() * (T * T).sum(0).sqrt() + 1e-8)


def episode(n_trials, mean_len, rng, sup_sec, n_query):
    """Chronological episode: ~sup_sec of consecutive support trials, gap 1, then n_query trials."""
    n_q = min(n_query, max(1, n_trials // 4))
    n_s = int(np.clip(round(sup_sec * FS / mean_len), 3, n_trials - n_q - 1))
    start = int(rng.integers(0, n_trials - n_s - n_q - 1 + 1))
    return list(range(start, start + n_s)), list(range(start + n_s + 1, start + n_s + 1 + n_q))


@torch.no_grad()
def val_score(enc, lam, subjects, dev, amp, max_test):
    enc.eval()
    rs = []
    for _, E, Y in subjects:
        cal, test = chrono_split(len(E), 0.2, gap=1)
        Xs, Ys = rows(enc, E, Y, cal, dev, amp)
        Xq, Yq = rows(enc, E, Y, test[:max_test], dev, amp)
        rs.append(col_r(ridge_torch(Xs, Ys, Xq, lam), Yq).mean().item())
    return float(np.mean(rs))


# ----------------------------------------------------------------------------- training
def train_encoder(a, train, val, dev, log, ckpt_path):
    enc = TemporalEncoder(**enc_cfg(a)).to(dev)
    log_lam = nn.Parameter(torch.tensor(float(np.log(a.lam0)), device=dev))
    opt = torch.optim.AdamW([{"params": enc.parameters(), "lr": a.lr, "weight_decay": 0.0},
                             {"params": [log_lam], "lr": a.lr_lam, "weight_decay": 0.0}])
    amp = a.amp and dev.type == "cuda"
    rng = np.random.default_rng(a.seed)
    mean_len = {sid: float(np.mean([e.shape[1] for e in E])) for sid, E, _ in train}
    step, best, best_state, bad, hist = 0, -1e9, None, 0, []
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=dev, weights_only=False)
        enc.load_state_dict(ck["enc"])
        opt.load_state_dict(ck["opt"])
        with torch.no_grad():
            log_lam.copy_(ck["log_lam"])
        step, best, best_state, bad, hist = ck["step"], ck["best"], ck["best_state"], ck["bad"], ck["hist"]
        rng = np.random.default_rng(a.seed + step)
        log(f"  resumed at step {step} (best val r={best:.4f})")
    else:
        r0 = val_score(enc, log_lam.exp().item(), val, dev, amp, a.val_trials)
        hist.append((0, r0))
        log(f"  step     0 | val r (Morlet init) = {r0:.4f}")
        best, best_state = r0, (copy.deepcopy(enc.state_dict()), log_lam.exp().item())
    t0, run_r = time.time(), []
    while step < a.steps and bad < a.patience:
        enc.train()
        loss = 0.0
        for si in rng.choice(len(train), a.subj_per_step, replace=len(train) < a.subj_per_step):
            sid, E, Y = train[si]
            sup, qry = episode(len(E), mean_len[sid], rng, a.sup_sec, a.n_query)
            Xs, Ys = rows(enc, E, Y, sup, dev, amp)
            Xq, Yq = rows(enc, E, Y, qry, dev, amp)
            r = col_r(ridge_torch(Xs, Ys, Xq, log_lam.exp()), Yq).mean()
            loss = loss + (1 - r) / a.subj_per_step
            run_r.append(r.item())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
        opt.step()
        step += 1
        if step % a.eval_every == 0 or step == a.steps:
            vr = val_score(enc, log_lam.exp().item(), val, dev, amp, a.val_trials)
            hist.append((step, vr))
            if vr > best:
                best, bad, best_state = vr, 0, (copy.deepcopy(enc.state_dict()), log_lam.exp().item())
            else:
                bad += 1
            log(f"  step {step:5d} | train query r={np.mean(run_r):.4f} | val r={vr:.4f} | "
                f"lam={log_lam.exp().item():.3g} | {time.time() - t0:.0f}s" + (" *" if bad == 0 else ""))
            run_r, t0 = [], time.time()
            if ckpt_path:
                torch.save({"enc": enc.state_dict(), "opt": opt.state_dict(), "log_lam": log_lam.detach(),
                            "step": step, "best": best, "best_state": best_state, "bad": bad, "hist": hist},
                           ckpt_path + ".tmp")
                os.replace(ckpt_path + ".tmp", ckpt_path)
    enc.load_state_dict(best_state[0])
    return enc, best_state[1], best, hist


def enc_cfg(a):
    return dict(n_pairs=a.n_pairs, half=a.half, fmin=a.fmin, fmax=a.fmax, cycles=a.cycles,
                n_free=a.n_free, envelope=not a.no_env)


# ----------------------------------------------------------------------------- evaluation
def np_feat(enc, dev, amp):
    """Encoder as a numpy feature function (C, L) -> (C*F, L) for the StreamRidge machinery."""
    enc = enc.eval()

    @torch.no_grad()
    def f(e):
        x = torch.from_numpy(np.ascontiguousarray(e, np.float32))[None].to(dev)
        with torch.autocast(device_type=dev.type, enabled=amp):
            h = enc(x)[0]
        return h.float().reshape(-1, h.shape[-1]).cpu().numpy()
    return f


_LINE = butter(4, (46, 54), btype="bandstop", fs=FS, output="sos")


def r_without_line(P, T):
    """Per-contact r after removing 46-54 Hz (mains residue) from BOTH prediction and target."""
    fp = [sosfiltfilt(_LINE, p, axis=1) for p in P]
    ft = [sosfiltfilt(_LINE, t, axis=1) for t in T]
    return per_contact_metrics(fp, ft, FS, bands=False)["r"]


def r_single_trial(P, T):
    """
    Per-contact r of the single-trial parts: each side minus ITS OWN trial average over the test
    trials (prediction minus the mean prediction, target minus the mean target). Removing the same
    template from both sides would add a shared component and inflate r.
    """
    tP, tT = template(P), template(T)
    return per_contact_metrics([p - tmpl_at(tP, p.shape[1]) for p in P],
                               [t - tmpl_at(tT, t.shape[1]) for t in T], FS, bands=False)["r"]


def amplitude_calibrate(kf, Yc, P):
    """
    Per-contact least-squares gain on the OUT-OF-FOLD calibration predictions. Alpha is chosen by
    r, which ignores amplitude; this fixes the amplitude (R^2) without changing r (gain >= 0).
    """
    ym = kf.s["ym"][:, None]
    num = sum(((o - ym) * (y - ym)).sum(1) for o, y in zip(kf.oof, Yc))
    den = sum(((o - ym) ** 2).sum(1) for o in kf.oof)
    g = np.clip(num / np.maximum(den, 1e-12), 0, None)[:, None]
    return [(ym + g * (p - ym)).astype(np.float32) for p in P]


def evaluate(E, Y, enc0, enc, a, dev, log):
    amp = a.amp and dev.type == "cuda"
    n = len(E)
    _, test = chrono_split(n, 0.2, gap=1)
    Et, Yt = [E[i] for i in test], [Y[i] for i in test]
    f0, f1 = np_feat(enc0, dev, amp), np_feat(enc, dev, amp)
    lin = RandomConvFeatures(E[0].shape[0], M=512, seed=0, mode="linear")
    res = {"n_trials": n, "fracs": {}}
    for f in a.fracs:
        n_c = max(3, int(round(n * f)))
        if n_c >= test[0]:
            continue
        Ec, Yc = E[:n_c], Y[:n_c]
        bands = abs(f - 0.2) < 1e-9
        out, on = {}, {}

        def score(name, P):
            m = per_contact_metrics(P, Yt, FS, bands=bands)
            m["r_no50"] = r_without_line(P, Yt)
            m["r_induced"] = r_single_trial(P, Yt)
            out[name] = {"mean": {k: float(np.nanmean(v)) for k, v in m.items()}, "r": [float(x) for x in m["r"]],
                         **{x: [float(v) for v in m[x]] for x in EXTRA_METRICS}}

        score("ridge", StreamRidge(LAG9).fit(Ec, Yc).predict(Et))
        base = KFoldRidge(LAG9, shared=True).fit(Ec, Yc)
        score("base", base.predict(Et))
        if not a.fast_eval:
            for name, feat in [("lin_gated", lin), ("enc0_gated", f0), ("enc_gated", f1)]:
                P, on[name] = gated(base, Ec, Yc, Et, feat)
                score(name, P)
        for name, feat in [("lin_ridge", lin)] * (not a.fast_eval) + [("enc0_ridge", f0)]:
            score(name, KFoldRidge(feat, ALPHAS_WIDE, shared=True, keep_oof=False).fit(Ec, Yc).predict(Et))
        kf = KFoldRidge(f1, ALPHAS_WIDE, shared=True).fit(Ec, Yc)
        P = kf.predict(Et)
        score("enc_ridge", P)
        score("enc_ridge_cal", amplitude_calibrate(kf, Yc, P))
        mins = sum(e.shape[1] for e in Ec) / FS / 60
        res["fracs"][f"{f:g}"] = {"n_calib": n_c, "minutes": mins, "gate_on": on,
                                  "mean": {k: v["mean"]["r"] for k, v in out.items()},
                                  **{x: {k: v[x] for k, v in out.items()} for x in EXTRA_METRICS},
                                  "bands": {k: v["mean"] for k, v in out.items()} if bands else {},
                                  "r": {k: v["r"] for k, v in out.items()}}
        log(f"    f={f:g} ({mins:.1f} min) " + " ".join(f"{k}={v['mean']['r']:+.3f}" for k, v in out.items()) +
            " | gate on: " + " ".join(f"{k}={v:.2f}" for k, v in on.items()))
    return res


KEY_TESTS = [("enc_ridge", "enc0_ridge", "learning helps (vs same encoder untrained)?"),
             ("enc_ridge", "lin_ridge", "learned filters beat random filters?"),
             ("enc0_ridge", "lin_ridge", "Morlet init beats random filters?"),
             ("enc_ridge", "base", "beats the strong linear baseline?"),
             ("lin_ridge", "base", "single-stage filter bank beats LAG9?")]


def key_tests(results, out_dir, log):
    """Paired subject-level tests for the decisive comparisons, on r and on every EXTRA_METRICS."""
    from scipy.stats import wilcoxon
    lines, js = ["\n### key comparisons (paired over subjects)",
                 "| calib | question | A | B | metric | mean A | mean B | A - B | A wins | Wilcoxon p |",
                 "|---|---|---|---|---|---|---|---|---|---|"], []
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r for r in results if f in r["fracs"] and "r_no50" in r["fracs"][f]]
        metrics = ("r",) + tuple(x for x in EXTRA_METRICS if rows and x in rows[0]["fracs"][f])
        for ma, mb, q in KEY_TESTS:
            if not rows or ma not in rows[0]["fracs"][f]["mean"] or mb not in rows[0]["fracs"][f]["mean"]:
                continue
            for metric in metrics:
                if metric == "r":
                    get = lambda r, m: r["fracs"][f]["mean"][m]                         # noqa: E731
                else:
                    get = lambda r, m: float(np.nanmean(r["fracs"][f][metric][m]))     # noqa: E731
                x = np.array([get(r, ma) for r in rows])
                y = np.array([get(r, mb) for r in rows])
                try:
                    p = float(wilcoxon(x, y).pvalue)
                except ValueError:
                    p = float("nan")
                js.append({"frac": f, "a": ma, "b": mb, "metric": metric, "mean_a": float(x.mean()),
                           "mean_b": float(y.mean()), "diff": float((x - y).mean()), "wins": int((x > y).sum()),
                           "n": len(x), "p": p})
                lines.append(f"| {f} | {q} | {ma} | {mb} | {metric} | {x.mean():.3f} | {y.mean():.3f} | "
                             f"{(x - y).mean():+.4f} | {(x > y).sum()}/{len(x)} | {p:.4f} |")
    md = "\n".join(lines)
    log(md)
    with open(os.path.join(out_dir, "summary.md"), "a", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "key_tests.json"), "w") as fh:
        json.dump(js, fh, indent=1)


def train_final(a, subjects, dev, log):
    """
    External-validation encoder: the leave-one-patient-out procedure with no test patient. The validation patients
    are drawn exactly as in a LOSO fold (generator seeded with --seed, first n_val_subj of a permutation); they are
    used only for early stopping / checkpoint selection, and the encoder is not refitted on them.
    """
    torch.manual_seed(a.seed)
    order = np.random.default_rng(a.seed).permutation(len(subjects))
    val = [subjects[i] for i in order[:a.n_val_subj]]
    train = [subjects[i] for i in order[a.n_val_subj:]]
    log("=" * 78)
    log(f"FINAL encoder | optimisation ({len(train)}): {[s[0] for s in train]} | "
        f"early stopping ({len(val)}): {[s[0] for s in val]}")
    t0 = time.time()
    ckpt = os.path.join(a.out, "ckpt_final.pt")
    enc, lam, best, hist = train_encoder(a, train, val, dev, log, ckpt)
    path = os.path.join(a.out, "enc_final.pt")
    torch.save({"state": enc.state_dict(), "cfg": enc.cfg, "lam": lam, "best_val": best, "hist": hist,
                "train_subjects": [s[0] for s in train], "val_subjects": [s[0] for s in val], "seed": a.seed,
                "config": vars(a)}, path + ".tmp")
    os.replace(path + ".tmp", path)
    if os.path.exists(ckpt):
        os.remove(ckpt)
    best_step = max(hist, key=lambda h: h[1])[0]
    log(f"final encoder: val r {hist[0][1]:.4f} (init) -> {best:.4f} (best, step {best_step}), last step "
        f"{hist[-1][0]}, lam={lam:.3g}, {enc.n_feat} features/channel | {(time.time() - t0) / 60:.1f} min -> {path}")


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--subjects", nargs="*", default=None, help="subset of subjects to load")
    ap.add_argument("--folds", nargs="*", default=None, help="test subjects to run (default: all)")
    ap.add_argument("--min_trials", type=int, default=20)
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.1, 0.2])
    # encoder
    ap.add_argument("--n_pairs", type=int, default=12)
    ap.add_argument("--half", type=int, default=240, help="filter half-length in samples (240 = 1.2 s)")
    ap.add_argument("--fmin", type=float, default=2.0)
    ap.add_argument("--fmax", type=float, default=60.0)
    ap.add_argument("--cycles", type=float, default=5.0)
    ap.add_argument("--n_free", type=int, default=4)
    ap.add_argument("--no_env", action="store_true")
    # episodic training
    ap.add_argument("--steps", type=int, default=3000)
    ap.add_argument("--eval_every", type=int, default=150)
    ap.add_argument("--patience", type=int, default=6)
    ap.add_argument("--subj_per_step", type=int, default=2)
    ap.add_argument("--sup_sec", type=float, default=240.0, help="support size (seconds), ~4 min as at test")
    ap.add_argument("--n_query", type=int, default=8)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--lr_lam", type=float, default=1e-2)
    ap.add_argument("--lam0", type=float, default=1.0)
    ap.add_argument("--n_val_subj", type=int, default=2)
    ap.add_argument("--val_trials", type=int, default=40, help="test trials per validation subject")
    ap.add_argument("--no_amp", action="store_true")
    ap.add_argument("--fast_eval", action="store_true",
                    help="only ridge, base, enc0_ridge, enc_ridge, enc_ridge_cal (for seed / ablation runs)")
    ap.add_argument("--reeval", action="store_true",
                    help="redo the evaluation of finished folds with their saved encoders (no retraining)")
    ap.add_argument("--final", action="store_true",
                    help="train ONE encoder on all subjects (external validation): n_val_subj subjects drawn with "
                         "--seed for early stopping, the rest for optimisation -> OUT/enc_final.pt; no evaluation")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    a.amp = not a.no_amp
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "shared.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    log(f"config: {vars(a)}")
    if a.final and os.path.exists(os.path.join(a.out, "enc_final.pt")):
        log("final encoder already trained: " + os.path.join(a.out, "enc_final.pt"))
        return
    subjects = []
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        sid = os.path.basename(p)[:-3]
        if a.subjects and sid not in a.subjects:
            continue
        s, E, Y = load_trials(p, "bip", "car")
        if len(E) < a.min_trials or s.n_targets == 0:
            log(f"skip {sid}: {len(E)} trials, {s.n_targets} targets")
            continue
        subjects.append((sid, E, Y))
        log(f"{sid}: {len(E)} trials, {E[0].shape[0]} scalp, {s.n_targets} targets")
    if a.final:
        train_final(a, subjects, dev, log)
        return
    folds = [s for s in subjects if not a.folds or s[0] in a.folds]

    results = []
    for fi, (sid, E, Y) in enumerate(folds):
        fp = os.path.join(a.out, f"fold_{sid}.json")
        if os.path.exists(fp):
            done = json.load(open(fp))
            # --reeval redoes only folds evaluated by an older version (missing a metric), so it can resume
            if not a.reeval or all(x in fr for fr in done["fracs"].values() for x in EXTRA_METRICS):
                results.append(done)
                log(f"[{sid}] already done, loaded")
                continue
        torch.manual_seed(a.seed + fi)
        log("=" * 78)
        log(f"[{fi + 1}/{len(folds)}] TEST {sid}")
        pool = [s for s in subjects if s[0] != sid]
        order = np.random.default_rng(a.seed + fi).permutation(len(pool))
        val = [pool[i] for i in order[:a.n_val_subj]]
        train = [pool[i] for i in order[a.n_val_subj:]]
        log(f"  train={[s[0] for s in train]} val={[s[0] for s in val]}")
        t0 = time.time()
        enc_path = os.path.join(a.out, f"enc_{sid}.pt")
        if os.path.exists(enc_path):
            ck = torch.load(enc_path, map_location=dev, weights_only=False)
            enc = TemporalEncoder(**ck["cfg"]).to(dev)
            enc.load_state_dict(ck["state"])
            lam, best, hist = ck["lam"], ck["best_val"], ck["hist"]
            log(f"  loaded trained encoder (val r={best:.4f})")
        else:
            ckpt = os.path.join(a.out, f"ckpt_{sid}.pt")
            enc, lam, best, hist = train_encoder(a, train, val, dev, log, ckpt)
            torch.save({"state": enc.state_dict(), "cfg": enc.cfg, "lam": lam, "best_val": best, "hist": hist},
                       enc_path)
            if os.path.exists(ckpt):
                os.remove(ckpt)
        enc0 = TemporalEncoder(**enc.cfg).to(dev)
        log(f"  encoder: val r {hist[0][1]:.4f} (init) -> {best:.4f} (trained), lam={lam:.3g}, "
            f"{enc.n_feat} features/channel | {time.time() - t0:.0f}s")
        r = evaluate(E, Y, enc0, enc, a, dev, log)
        r.update(subject=sid, val_subjects=[s[0] for s in val], val_r_init=hist[0][1], val_r=best, lam=lam,
                 minutes=(time.time() - t0) / 60)
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
        del enc, enc0
        if dev.type == "cuda":
            torch.cuda.empty_cache()
    if results:
        summarize(results, a.out, log)
        key_tests(results, a.out, log)
        vi = [r["val_r_init"] for r in results]
        vt = [r["val_r"] for r in results]
        log(f"validation subjects, ridge readout on 20 %: Morlet init r={np.mean(vi):.4f} -> trained {np.mean(vt):.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/readout.py
"""
Phase 4 - readout improvements on the SAVED shared encoders (CPU, no retraining).

Phase 3 analysis of the shared encoder (15 subjects, 20 % calibration):
  (1) training traded high frequencies for low ones: the untrained Morlet bank (enc0) beats the
      trained encoder in alpha/beta, the trained one wins in delta/theta; two pairs collapsed on 50 Hz;
  (2) half of the features (pairs >= 13 Hz and their envelopes) sit in bands with little predictable
      signal, but one shared ridge penalty treats them like the useful ones;
  (3) at 1-2 min of calibration the readout (C x 40 weights per contact) is the bottleneck.
Every candidate below keeps the encoder fixed and changes only the readout:

  enc         reference = enc_ridge of s2i.shared (ridge, one K-fold alpha)
  base        reference = ridge_kfold_shared on LAG9
  enc_low     only low-frequency encoder features (pairs < 13 Hz, their envelopes, free filters)
  enc_group   high-frequency features down-weighted by a K-fold-chosen factor (two penalties)
  enc_enc0hi  + the UNTRAINED encoder's high-frequency features (alpha/beta), own K-fold weight
  enc_lag9    + LAG9 (raw lags +-200 ms), own K-fold weight
  enc_bands   target split into 5 complementary bands (<4, 4-8, 8-13, 13-30, >30 Hz); one ridge per
              band with its own K-fold penalty (chosen by MSE); predictions summed
  enc_sess    encoder features standardised per recording session (EEG only, label-free,
              transductive: uses the EEG of every trial of the session)
  enc_warp    filters time-scaled by s in {0.9, 1, 1.1} (frequencies x 1/s), s chosen per subject by
              K-fold (individual frequency adaptation)
  enc_bilin   structured readout W_k = a_k b_k^T (channels x features, rank 1-3, spectral profiles b
              orthonormal, ridge penalty on a), rank and penalty by K-fold: (C + F) R weights per contact
  ens_base    enc and base averaged with a K-fold-chosen weight
  ens_seed    enc of this run and of the seed-1 run averaged with a K-fold-chosen weight (if present)
--strict: undo preprocessing's whole-session normalisation and re-normalise EEG and target with the
calibration trials' statistics only; session-standardised models are skipped (combo_nosess = combo
without session standardisation is reported in every mode).
Phase 4b (--only_combo, combinations of the three Phase-4 winners):
  ens_all       ridge readouts of ALL available encoders (main + seed runs) averaged
  ens_all_sess  same, every encoder's features standardised per session
  combo         ens_all_sess mixed with base (LAG9 ridge), weight chosen by K-fold

Exactness: every readout is linear in its features, so K-fold out-of-fold r, the amplitude gain and
test r / R^2 are computed EXACTLY from second-order sufficient statistics (sums of x, xx', y, y^2,
xy') accumulated in one pass over the trials. `enc` therefore reproduces enc_ridge of s2i.shared.
Test trials, calibration fractions and fold blocks are those of phases 0-3.

Usage
-----
python -m s2i.readout --data DATA_DIR --run RUNS_DIR/shared_bip_car [--seed_run RUNS_DIR/shared_seed1]
python -m s2i.readout --data DATA_DIR --run RUNS_DIR/shared_bip_car --only_combo
       --seed_runs RUNS_DIR/shared_seed1 RUNS_DIR/shared_seed2     (Phase 4b: combinations of the winners)
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
import torch
from scipy.signal import butter, sosfiltfilt

from s2i.ceiling import ALPHAS, chrono_split, feat_lag
from s2i.baselines import DEFAULT_LAGS
from s2i.metrics import per_contact_metrics
from s2i.shared import TemporalEncoder, np_feat, r_single_trial, r_without_line
from s2i.within import ALPHAS_WIDE, _blocks

FS = 200
HIGH_HZ = 13.0
Y_CUTS = (4.0, 8.0, 13.0, 30.0)                       # -> 5 complementary target bands
GROUP_SCALES = (1.0, 0.5, 0.25, 0.1, 0.0)             # weight of the high-frequency group
ADD_SCALES = (0.0, 0.25, 0.5, 1.0, 2.0)               # weight of an added feature block
ENS_W = (0.0, 0.25, 0.5, 0.75, 1.0)                   # weight of enc in an ensemble
WARPS = (0.9, 1.1)
BILIN = tuple((r, al) for r in (1, 2, 3) for al in (0.001, 0.01, 0.1, 1.0, 10.0))
LAG9 = feat_lag(DEFAULT_LAGS)
_LP = {c: butter(4, c, btype="low", fs=FS, output="sos") for c in Y_CUTS}
_DEV = None                                           # set in main(); Gram matrices on the GPU if cuda


# ----------------------------------------------------------------------------- statistics
class St:
    """Second-order sufficient statistics of (features x, target y); optional band targets."""

    def __init__(self, D, K, nb=0, Db=0):
        self.n = 0.0
        self.sx, self.sxx = np.zeros(D), np.zeros((D, D))
        self.sy, self.syy, self.sxy = np.zeros(K), np.zeros(K), np.zeros((D, K))
        self.syb, self.syyb, self.sxyb = np.zeros((nb, K)), np.zeros((nb, K)), np.zeros((nb, Db, K))

    def add(self, X, y, yb=None):
        self.n += X.shape[1]
        self.sx += X.sum(1)
        if _DEV is not None and _DEV.type == "cuda":             # large Gram matrices on the GPU
            Xt, yt = torch.from_numpy(X).to(_DEV), torch.from_numpy(np.ascontiguousarray(y)).to(_DEV)
            self.sxx += (Xt @ Xt.T).double().cpu().numpy()
            self.sxy += (Xt @ yt.T).double().cpu().numpy()
        else:
            self.sxx += X @ X.T
            self.sxy += X @ y.T
        self.sy += y.sum(1)
        self.syy += (y * y).sum(1)
        if yb is not None:
            Xb = X[:self.sxyb.shape[1]]
            self.syb += yb.sum(2)
            self.syyb += (yb * yb).sum(2)
            for b in range(len(yb)):
                self.sxyb[b] += Xb @ yb[b].T

    def _new(self):
        return St.__new__(St)

    def __add__(self, o):
        s = self._new()
        for k in ("n", "sx", "sxx", "sy", "syy", "sxy", "syb", "syyb", "sxyb"):
            setattr(s, k, getattr(self, k) + getattr(o, k))
        return s

    def __sub__(self, o):
        s = self._new()
        for k in ("n", "sx", "sxx", "sy", "syy", "sxy", "syb", "syyb", "sxyb"):
            setattr(s, k, getattr(self, k) - getattr(o, k))
        return s

    def take(self, idx, band=None, transform=None):
        """Sub-block of features `idx` (band b replaces the target), optionally x -> (x - m) / d."""
        s = self._new()
        s.n, sx, sxx = self.n, self.sx[idx], self.sxx[np.ix_(idx, idx)]
        if band is None:
            s.sy, s.syy, sxy = self.sy, self.syy, self.sxy[idx]
        else:
            s.sy, s.syy, sxy = self.syb[band], self.syyb[band], self.sxyb[band][idx]
        if transform is not None:
            m, d = transform
            sxx = (sxx - np.outer(m, sx) - np.outer(sx, m) + s.n * np.outer(m, m)) / np.outer(d, d)
            sxy = (sxy - np.outer(m, s.sy)) / d[:, None]
            sx = (sx - s.n * m) / d
        s.sx, s.sxx, s.sxy = sx, sxx, sxy
        K = len(s.sy)
        s.syb, s.syyb, s.sxyb = np.zeros((0, K)), np.zeros((0, K)), np.zeros((0, len(idx), K))
        return s


def total(sts):
    out = sts[0]
    for s in sts[1:]:
        out = out + s
    return out


# ----------------------------------------------------------------------------- linear readouts
def standardise(tr):
    n = tr.n
    mu, ym = tr.sx / n, tr.sy / n
    G = tr.sxx - n * np.outer(mu, mu)
    sd = np.sqrt(np.clip(np.diag(G) / n, 0, None)) + 1e-8
    Gs = G / np.outer(sd, sd)
    Bs = (tr.sxy - n * np.outer(mu, ym)) / sd[:, None]
    return mu, sd, (Gs + Gs.T) / 2, Bs, ym


def to_raw(Ws, mu, sd, ym):
    A = Ws / sd[:, None]
    return A, ym - mu @ A


def ridge_models(tr, alphas, scale_vec=None):
    """Ridge for each alpha (penalty alpha * mean eigenvalue). scale_vec multiplies standardised features."""
    mu, sd, Gs, Bs, ym = standardise(tr)
    sc = max(np.trace(Gs) / len(Gs), 1e-12)      # penalty scale of the UNSCALED design (= n)
    if scale_vec is not None:
        Gs, Bs = Gs * np.outer(scale_vec, scale_vec), Bs * scale_vec[:, None]
    lam, Q = np.linalg.eigh(Gs)
    lam = np.clip(lam, 0, None)
    QB = Q.T @ Bs
    out = []
    for a in alphas:
        Ws = Q @ (QB / (lam + a * sc)[:, None])
        if scale_vec is not None:
            Ws = Ws * scale_vec[:, None]
        out.append(to_raw(Ws, mu, sd, ym))
    return out


def bilinear_models(tr, C, F, cands, iters=5):
    """
    W_k = a_k b_k^T over (channel, feature) with rank R: b_k (F x R) has orthonormal columns (spectral
    profiles), a_k (C x R) carries the ridge penalty alpha * n, exactly the ridge scale. Alternating
    least squares, initialised by the SVD of a ridge fit.
    """
    mu, sd, Gs, Bs, ym = standardise(tr)
    K, sc = Bs.shape[1], max(np.trace(Gs) / len(Gs), 1e-12)
    G4, B3 = Gs.reshape(C, F, C, F), Bs.reshape(C, F, K)
    lam, Q = np.linalg.eigh(Gs)
    W0 = Q @ ((Q.T @ Bs) / (np.clip(lam, 0, None) + sc)[:, None])
    _, _, Vt = np.linalg.svd(W0.T.reshape(K, C, F), full_matrices=False)
    out = []
    for R, alpha in cands:
        R_ = min(R, Vt.shape[1])
        b = Vt[:, :R_, :].transpose(0, 2, 1)                     # (K, F, R) orthonormal columns

        def solve_a(b):
            # Ga[k] = (I_C x b_k)' G (I_C x b_k): contract g with b (one matmul), then f (batched matmul)
            M = (Gs.reshape(C * F * C, F) @ b.transpose(1, 0, 2).reshape(F, K * R_)).reshape(C, F, C, K, R_)
            Ga = np.matmul(M.transpose(3, 0, 2, 4, 1), b[:, None, None]).transpose(0, 1, 4, 2, 3)
            ca = np.einsum("cfk,kfr->kcr", B3, b).reshape(K, C * R_, 1)
            return np.linalg.solve(Ga.reshape(K, C * R_, C * R_) + alpha * sc * np.eye(C * R_),
                                   ca)[..., 0].reshape(K, C, R_)
        a = solve_a(b)
        for _ in range(iters):
            # Gb[k] = (a_k x I_F)' G (a_k x I_F): contract c with a (one matmul), then d (batched matmul)
            M = (a.transpose(0, 2, 1).reshape(K * R_, C) @ Gs.reshape(C, F * C * F)).reshape(K, R_, F, C, F)
            Gb = np.matmul(M.transpose(0, 1, 2, 4, 3), a[:, None, None]).transpose(0, 2, 1, 3, 4)
            Gb = Gb.reshape(K, F * R_, F * R_)
            cb = np.einsum("cfk,kcr->kfr", B3, a).reshape(K, F * R_, 1)
            b = np.linalg.solve(Gb + 1e-4 * sc * np.eye(F * R_), cb)[..., 0].reshape(K, F, R_)
            b, _ = np.linalg.qr(b)                               # keep only the profile subspace
            a = solve_a(b)
        Ws = np.einsum("kcr,kfr->cfk", a, b).reshape(C * F, K)
        out.append(to_raw(Ws, mu, sd, ym))
    return out


def pred_sums(A, c, te):
    """Sums of prediction p = x'A + c over the samples summarised by `te`: (n, sp, spp, spy, sy, syy)."""
    Ax = A.T @ te.sx
    sp = Ax + te.n * c
    spp = ((te.sxx @ A) * A).sum(0) + 2 * c * Ax + te.n * c ** 2
    spy = (A * te.sxy).sum(0) + c * te.sy
    return np.stack([np.full_like(sp, te.n), sp, spp, spy, te.sy, te.syy])


def r_of(s):
    n, sp, spp, spy, sy, syy = s
    den = np.sqrt(np.clip(n * spp - sp ** 2, 0, None) * np.clip(n * syy - sy ** 2, 0, None))
    return np.where(den > 0, (n * spy - sp * sy) / np.maximum(den, 1e-12), np.nan)


def sse_of(s):
    return s[2] - 2 * s[3] + s[5]


class Fit:
    """K-fold selection over candidates from statistics. fit(train_stats) -> [(A, c)] per candidate."""

    def __init__(self, fit, folds, tot, crit="r"):
        self.fit, self.tot = fit, tot
        self.oof = None
        for tr, te in folds:
            s = np.stack([pred_sums(A, c, te) for A, c in fit(tr)])
            self.oof = s if self.oof is None else self.oof + s
        if crit == "r":
            score = [np.nanmean(r_of(o)) for o in self.oof]
            self.best = int(np.nanargmax(score))
        else:
            self.best = int(np.argmin([np.nansum(sse_of(o)) for o in self.oof]))
        self.A, self.c = fit(tot)[self.best]
        self.fold_models = None

    def fold_model(self, folds):
        if self.fold_models is None:
            self.fold_models = [self.fit(tr)[self.best] for tr, _ in folds]
        return self.fold_models


def calibrated(A, c, oof, ym):
    """Per-contact least-squares gain on the pooled out-of-fold predictions (r unchanged, R^2 honest)."""
    n, sp, spp, spy, sy, _ = oof
    num = spy - ym * sp - ym * sy + n * ym ** 2
    den = spp - 2 * ym * sp + n * ym ** 2
    g = np.clip(num / np.maximum(den, 1e-12), 0, None)
    return A * g, ym + g * (c - ym)


def score(A, c, oof, tot, te, te_sessions=None):
    ym = tot.sy / tot.n
    s = pred_sums(A, c, te)
    Ac, cc = calibrated(A, c, oof, ym)
    sc = pred_sums(Ac, cc, te)
    r2 = 1 - sse_of(sc) / np.maximum(sc[5] - sc[4] ** 2 / sc[0], 1e-12)
    out = {"r": float(np.nanmean(r_of(s))), "r2": float(np.nanmean(r2)), "contacts": [float(x) for x in r_of(s)],
           "cv_contacts": [float(x) for x in r_of(oof)]}          # K-fold r on calibration (reliability map)
    if te_sessions:                                            # calibration longevity: r per test session
        out["sessions"] = {k: float(np.nanmean(r_of(pred_sums(A, c, v)))) for k, v in te_sessions.items()}
    return out


def embed(A, c, idx, D):
    out = np.zeros((D, A.shape[1]))
    out[idx] = A
    return out, c


# ----------------------------------------------------------------------------- features
def load_encoder(path, dev):
    ck = torch.load(path, map_location=dev, weights_only=False)
    enc = TemporalEncoder(**ck["cfg"]).to(dev)
    enc.load_state_dict(ck["state"])
    return enc.eval()


def warped(enc, s, dev):
    """Same encoder with every kernel time-scaled by s: h_s(t) = h(s t), i.e. frequencies x 1/s."""
    out = TemporalEncoder(**enc.cfg).to(dev)
    w = enc.conv.weight.detach().cpu().numpy()[:, 0]
    H = w.shape[1] // 2
    t = np.arange(-H, H + 1)
    ws = np.stack([np.interp(t / s, t, k, left=0.0, right=0.0) for k in w])
    ws *= np.linalg.norm(w, axis=1, keepdims=True) / (np.linalg.norm(ws, axis=1, keepdims=True) + 1e-12)
    with torch.no_grad():
        out.conv.weight.copy_(torch.tensor(ws, dtype=torch.float32)[:, None])
    return out.eval()


def high_mask(enc, use_init):
    """Per-feature mask (length n_feat): True for quadrature pairs >= HIGH_HZ and their envelopes."""
    cfg = enc.cfg
    P = cfg["n_pairs"]
    if use_init:
        f = np.geomspace(cfg["fmin"], cfg["fmax"], P)
    else:
        w = enc.conv.weight.detach().cpu().numpy()[:, 0]
        spec = np.abs(np.fft.rfft(w, 4096, axis=1)) ** 2
        fr = np.fft.rfftfreq(4096, 1 / FS)
        f = fr[np.argmax(spec[0:2 * P:2] + spec[1:2 * P:2], 1)]
    m = np.zeros(enc.n_feat, bool)
    for p in np.where(f >= HIGH_HZ)[0]:
        m[[2 * p, 2 * p + 1]] = True
        if cfg["envelope"]:
            m[2 * P + cfg["n_free"] + p] = True
    return m, f


def band_targets(y):
    lp = [sosfiltfilt(_LP[c], y, axis=1) for c in Y_CUTS]
    parts = [lp[0]] + [lp[i] - lp[i - 1] for i in range(1, len(lp))] + [y - lp[-1]]
    return np.stack(parts).astype(np.float32)


# ----------------------------------------------------------------------------- per subject
def load_subject(path):
    """-> trials of CAR EEG, trials of bipolar targets, trial sessions, stimulus events (or None)."""
    d = torch.load(path, weights_only=False)
    eeg, tgt = d["eeg_car"].half().float().numpy(), d["ieeg_bip"].half().float().numpy()
    b = d["trial_bounds"].numpy()
    ses = list(d.get("trial_session", ["ses-01"] * len(b)))
    ev = d.get("stim_events")
    return ([eeg[:, s0:s1] for s0, s1 in b], [tgt[:, s0:s1] for s0, s1 in b], ses,
            None if ev is None else np.asarray(ev, int).reshape(-1, 2))


EPOCH = (-100, 500)                                   # -0.5 to +2.5 s around a stimulus marker (samples)


def r_event(P, T, test, events):
    """
    Per-contact r without the stimulus-locked component: each side minus ITS OWN average over the test stimulus
    epochs (-0.5 to +2.5 s around each stimulus marker). events: rows (trial index, onset sample within the trial).
    An epoch is used only if it lies inside one test trial / window; a sample covered by two epochs takes the
    later one. Returns NaN per contact if there is no such epoch.
    """
    pos = {int(w): j for j, w in enumerate(test)}
    ep = sorted((pos[int(w)], int(o)) for w, o in events
                if int(w) in pos and o + EPOCH[0] >= 0 and o + EPOCH[1] <= P[pos[int(w)]].shape[1])
    if not ep:
        return np.full(P[0].shape[0], np.nan)
    tP = np.mean([P[j][:, o + EPOCH[0]:o + EPOCH[1]] for j, o in ep], 0)
    tT = np.mean([T[j][:, o + EPOCH[0]:o + EPOCH[1]] for j, o in ep], 0)
    Pr, Tr = [p.astype(np.float64) for p in P], [t.astype(np.float64) for t in T]
    for j, o in ep:                                   # ascending: a later epoch overwrites the overlap
        Pr[j][:, o + EPOCH[0]:o + EPOCH[1]] = P[j][:, o + EPOCH[0]:o + EPOCH[1]] - tP
        Tr[j][:, o + EPOCH[0]:o + EPOCH[1]] = T[j][:, o + EPOCH[0]:o + EPOCH[1]] - tT
    return per_contact_metrics(Pr, Tr, FS, bands=False)["r"]


def _robust(x):
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) * 1.4826
    return med, np.where(mad < 1e-12, 1.0, mad)


def strict_renorm(E, Y, ses, stats, n_cal):
    """
    Undo the whole-session robust normalisation of preprocessing (x * MAD_s + median_s) and re-normalise every
    trial with the median / MAD of the CALIBRATION trials only (first n_cal), for EEG and target alike:
    no statistic of a test trial is used anywhere (strict, deployment-faithful normalisation).
    """
    def redo(X, key):
        raw = []
        for x, s in zip(X, ses):
            st = stats["norm_stats"][key][s]
            raw.append(x * np.asarray(st["mad"], np.float32)[:, None] + np.asarray(st["med"], np.float32)[:, None])
        med, mad = _robust(np.concatenate(raw[:n_cal], 1))
        return [((r - med) / mad).astype(np.float32) for r in raw]
    return redo(E, "eeg_car"), redo(Y, "ieeg_bip")


def analyze(E, Y, ses, enc, seed_encs, a, dev, log, events=None):
    amp = dev.type == "cuda"
    zoo = not a.only_combo
    n, C, K = len(E), E[0].shape[0], Y[0].shape[0]
    _, test = chrono_split(n, 0.2, gap=1)
    enc0 = TemporalEncoder(**enc.cfg).to(dev).eval()
    F = enc.n_feat
    hi_enc, f_enc = high_mask(enc, use_init=False)
    hi_e0, _ = high_mask(enc0, use_init=True)
    f1, f0 = np_feat(enc, dev, amp), np_feat(enc0, dev, amp)
    fss = [np_feat(se, dev, amp) for se in seed_encs]
    fs1 = fss[0] if fss else None
    fw = {s: np_feat(warped(enc, s, dev), dev, amp) for s in WARPS} if zoo else {}
    e0_rows = np.where(np.tile(hi_e0, C))[0]
    seeds = [f"seed{j}" for j in range(len(fss))]

    # main view = [enc | enc0 high-frequency rows (zoo only) | LAG9 | seed encoders]
    blocks = [("enc", C * F)] + ([("enc0hi", len(e0_rows))] if zoo else []) + [("lag9", C * len(DEFAULT_LAGS))]
    blocks += [(b, C * se.n_feat) for b, se in zip(seeds, seed_encs)]     # encoders may differ (e.g. no_env)
    idx, o = {}, 0
    for name, size in blocks:
        idx[name] = np.arange(o, o + size)
        o += size
    D = o
    hi_rows = np.where(np.tile(hi_enc, C))[0]
    lo_rows = np.where(~np.tile(hi_enc, C))[0]

    def main_x(e):
        x1 = f1(e)
        parts = [x1] + ([f0(e)[e0_rows]] if zoo else []) + [LAG9(e)] + [f(e) for f in fss]
        return np.concatenate(parts).astype(np.float32), x1

    # session statistics of every encoder's features (EEG of every trial, no target)
    enc_fns = {"enc": f1, **dict(zip(seeds, fss))}
    sess_sum = {}
    for i in range(n):
        for b, fn in enc_fns.items():
            x = fn(E[i]).astype(np.float64)
            s = sess_sum.setdefault((b, ses[i]), [0, np.zeros(len(idx[b])), np.zeros(len(idx[b]))])
            s[0] += x.shape[1]
            s[1] += x.sum(1)
            s[2] += (x * x).sum(1)
    sess_tf = {k: (v[1] / v[0], np.sqrt(np.clip(v[2] / v[0] - (v[1] / v[0]) ** 2, 0, None)) + 1e-6)
               for k, v in sess_sum.items()}

    def tf_for(names, session):
        """Per-session standardisation of encoder blocks; identity for other blocks (e.g. LAG9)."""
        m = [sess_tf[(b, session)][0] if (b, session) in sess_tf else np.zeros(len(idx[b])) for b in names]
        d = [sess_tf[(b, session)][1] if (b, session) in sess_tf else np.ones(len(idx[b])) for b in names]
        return np.concatenate(m), np.concatenate(d)

    # test statistics (per session, for the session-standardised readout)
    te_main, te_w = {}, {s: St(C * F, K) for s in fw}
    for i in test:
        X, _ = main_x(E[i])
        te_main.setdefault(ses[i], St(D, K)).add(X, Y[i])
        for s in fw:
            te_w[s].add(fw[s](E[i]).astype(np.float32), Y[i])
    te = total(list(te_main.values()))
    te_enc = te.take(idx["enc"])
    te_sess = total([v.take(idx["enc"], transform=tf_for(["enc"], k)) for k, v in te_main.items()])

    order = list(dict.fromkeys(ses))                     # sessions in recording order
    res = {"n_trials": n, "n_test": len(test), "n_sessions": len({k[1] for k in sess_sum}),
           "session_order": order, "test_trials_per_session": {k: sum(ses[i] == k for i in test) for k in order},
           "n_encoders": 1 + len(fss),
           "pair_peak_hz": [float(x) for x in f_enc], "fracs": {}}
    for frac in a.fracs:
        n_c = max(3, int(round(n * frac)))
        if n_c >= test[0]:
            continue
        t0 = time.time()
        fold_ids = _blocks(n_c, 5)
        cal = {}                                 # (fold, session) -> St (main view, with band targets)
        cal_w = {s: [St(C * F, K) for _ in fold_ids] for s in fw}
        nbt = len(Y_CUTS) + 1 if zoo else 0
        for fi, ids in enumerate(fold_ids):
            for i in ids:
                X, _ = main_x(E[i])
                cal.setdefault((fi, ses[i]), St(D, K, nb=nbt, Db=C * F if zoo else 0)).add(
                    X, Y[i], band_targets(Y[i]) if zoo else None)
                for s in fw:
                    cal_w[s][fi].add(fw[s](E[i]).astype(np.float32), Y[i])
        fold_st = [total([v for (f, _), v in cal.items() if f == fi]) for fi in range(len(fold_ids))]
        tot = total(fold_st)

        def folds_of(take):
            return [(take(tot - fs), take(fs)) for fs in fold_st]

        out, extra = {}, {}

        def run(name, take, fit, test_st, crit="r", note=None):
            fl = folds_of(take)
            m = Fit(fit, fl, take(tot), crit)
            out[name] = score(m.A, m.c, m.oof[m.best], take(tot), test_st,
                              {k: take(v) for k, v in te_main.items()})
            if note:
                extra[name] = note(m)
            return m, fl

        t_enc = lambda s: s.take(idx["enc"])                                    # noqa: E731
        m_enc, fl_enc = run("enc", t_enc, lambda tr: ridge_models(tr, ALPHAS_WIDE), te_enc,
                            note=lambda m: ALPHAS_WIDE[m.best])
        t_lag = lambda s: s.take(idx["lag9"])                                   # noqa: E731
        m_base, fl_base = run("base", t_lag, lambda tr: ridge_models(tr, ALPHAS), te.take(idx["lag9"]))
        store = {}                       # model -> (feature rows, calibrated A, c, session-standardised blocks)
        for nm, m_, rows in (("enc", m_enc, idx["enc"]), ("base", m_base, idx["lag9"])):
            ts = tot.take(rows)
            store[nm] = (rows, *calibrated(m_.A, m_.c, m_.oof[m_.best], ts.sy / ts.n), None)

        if zoo:
            lo = idx["enc"][lo_rows]
            run("enc_low", lambda s: s.take(lo), lambda tr: ridge_models(tr, ALPHAS_WIDE), te.take(lo))

            gmask = np.zeros(C * F, bool)
            gmask[hi_rows] = True
            cands = [(g, al) for g in GROUP_SCALES for al in ALPHAS_WIDE]

            def fit_group(tr):
                outm = []
                for g in GROUP_SCALES:
                    outm += ridge_models(tr, ALPHAS_WIDE, np.where(gmask, g, 1.0))
                return outm
            run("enc_group", t_enc, fit_group, te_enc, note=lambda m: cands[m.best][0])

            def added(name, block):
                sel = np.concatenate([idx["enc"], idx[block]])
                mask = np.r_[np.zeros(len(idx["enc"]), bool), np.ones(len(idx[block]), bool)]
                cands_ = [(g, al) for g in ADD_SCALES for al in ALPHAS_WIDE]

                def fit(tr):
                    outm = []
                    for g in ADD_SCALES:
                        outm += ridge_models(tr, ALPHAS_WIDE, np.where(mask, g, 1.0))
                    return outm
                run(name, lambda s: s.take(sel), fit, te.take(sel), note=lambda m: cands_[m.best][0])
            added("enc_enc0hi", "enc0hi")
            added("enc_lag9", "lag9")

            # one ridge per target band (penalty by MSE on that band), predictions summed
            nb = len(Y_CUTS) + 1
            bm = [Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), folds_of(lambda s, b=b: s.take(idx["enc"], band=b)),
                      tot.take(idx["enc"], band=b), crit="mse") for b in range(nb)]
            A = sum(m.A for m in bm)
            c = sum(m.c for m in bm)
            fms = [m.fold_model([(tr, None) for tr, _ in folds_of(lambda s, b=b: s.take(idx["enc"], band=b))])
                   for b, m in enumerate(bm)]
            oof = sum(pred_sums(sum(fms[b][fi][0] for b in range(nb)), sum(fms[b][fi][1] for b in range(nb)), te_f)
                      for fi, (_, te_f) in enumerate(fl_enc))
            out["enc_bands"] = score(A, c, oof, t_enc(tot), te_enc)
            extra["enc_bands"] = [ALPHAS_WIDE[m.best] for m in bm]

            # per-session standardisation of the encoder features
            sess_tot = total([v.take(idx["enc"], transform=tf_for(["enc"], s)) for (f, s), v in cal.items()])
            sess_folds = [total([v.take(idx["enc"], transform=tf_for(["enc"], s)) for (f, s), v in cal.items()
                                 if f == fi])
                          for fi in range(len(fold_ids))]
            m = Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), [(sess_tot - fs, fs) for fs in sess_folds], sess_tot)
            out["enc_sess"] = score(m.A, m.c, m.oof[m.best], sess_tot, te_sess)

            # individual frequency adaptation: pick s in {0.9, 1, 1.1} by K-fold r
            best = ("1.0", m_enc, np.nanmean(r_of(m_enc.oof[m_enc.best])), te_enc, t_enc(tot))
            for s in WARPS:
                tw = total(cal_w[s])
                mw = Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), [(tw - fs, fs) for fs in cal_w[s]], tw)
                sw = np.nanmean(r_of(mw.oof[mw.best]))
                if sw > best[2]:
                    best = (f"{s:g}", mw, sw, te_w[s], tw)
            out["enc_warp"] = score(best[1].A, best[1].c, best[1].oof[best[1].best], best[4], best[3])
            extra["enc_warp"] = best[0]

            # structured (bilinear) readout
            if not a.no_bilin:
                run("enc_bilin", t_enc, lambda tr: bilinear_models(tr, C, F, BILIN), te_enc,
                    note=lambda m: list(BILIN[m.best]))

        # ensembles: enc + base, enc + seed-1 (weights chosen by pooled out-of-fold r)
        def ensemble(name, m1, fl1, i1, m2, fl2, i2):
            sel = np.concatenate([i1, i2])
            n1 = len(i1)
            fm1 = m1.fold_model(fl1)
            fm2 = m2.fold_model(fl2)
            t_sel = lambda s: s.take(sel)                                       # noqa: E731
            fl = folds_of(t_sel)
            oofs = []
            for w in ENS_W:
                oofs.append(sum(pred_sums(np.r_[w * fm1[fi][0], (1 - w) * fm2[fi][0]], w * fm1[fi][1] +
                                          (1 - w) * fm2[fi][1], te_f) for fi, (_, te_f) in enumerate(fl)))
            wi = int(np.nanargmax([np.nanmean(r_of(o)) for o in oofs]))
            w = ENS_W[wi]
            A_ = np.r_[w * m1.A, (1 - w) * m2.A]
            out[name] = score(A_, w * m1.c + (1 - w) * m2.c, oofs[wi], t_sel(tot), te.take(sel))
            extra[name] = w
            assert len(A_) == len(sel) and n1 == len(m1.A)
        ensemble("ens_base", m_enc, fl_enc, idx["enc"], m_base, fl_base, idx["lag9"])
        if fss:
            t_seed = lambda s: s.take(idx["seed0"])                             # noqa: E731
            m_seed, fl_seed = run("seed", t_seed, lambda tr: ridge_models(tr, ALPHAS_WIDE), te.take(idx["seed0"]))
            ensemble("ens_seed", m_enc, fl_enc, idx["enc"], m_seed, fl_seed, idx["seed0"])

        # combinations of the three Phase-4 winners: several encoders averaged (ens_seed), session
        # standardisation (enc_sess) and mixing with LAG9 ridge (ens_base); exact from statistics
        def mixture(name, encs, sess, fixed_w=None):
            names = encs + ["lag9"]
            sel = np.concatenate([idx[b] for b in names])

            def T(st, session):
                return st.take(sel, transform=tf_for(names, session) if sess else None)
            folds_T = [total([T(v, ss) for (f, ss), v in cal.items() if f == fi]) for fi in range(len(fold_ids))]
            tot_T = total(folds_T)
            te_T = total([T(v, ss) for ss, v in te_main.items()])
            comps, o_ = [], 0
            for b in names:
                loc = np.arange(o_, o_ + len(idx[b]))
                o_ += len(idx[b])
                al = ALPHAS if b == "lag9" else ALPHAS_WIDE
                fl = [((tot_T - fs).take(loc), fs.take(loc)) for fs in folds_T]
                m = Fit(lambda tr, al=al: ridge_models(tr, al), fl, tot_T.take(loc))
                comps.append((b, loc, m, m.fold_model(fl)))

            def combine(w, models):
                A_, c_ = np.zeros((len(sel), K)), np.zeros(K)
                for (b, loc, _, _), (Ab, cb) in zip(comps, models):
                    wt = (1 - w) if b == "lag9" else w / len(encs)
                    A_[loc] += wt * Ab
                    c_ += wt * cb
                return A_, c_
            ws = ENS_W if fixed_w is None else (fixed_w,)
            oofs = [sum(pred_sums(*combine(w, [cm[3][fi] for cm in comps]), fs) for fi, fs in enumerate(folds_T))
                    for w in ws]
            wi = int(np.nanargmax([np.nanmean(r_of(o)) for o in oofs]))
            A_, c_ = combine(ws[wi], [(cm[2].A, cm[2].c) for cm in comps])
            out[name] = score(A_, c_, oofs[wi], tot_T, te_T, {ss: T(v, ss) for ss, v in te_main.items()})
            store[name] = (sel, *calibrated(A_, c_, oofs[wi], tot_T.sy / tot_T.n), names if sess else None)
            extra[name] = ws[wi]

        all_encs = ["enc"] + seeds
        if not zoo and not a.strict:
            mixture("enc_sess", ["enc"], sess=True, fixed_w=1.0)
        if seeds:
            mixture("ens_all", all_encs, sess=False, fixed_w=1.0)
            if not a.strict:
                mixture("ens_all_sess", all_encs, sess=True, fixed_w=1.0)
        if not a.strict:                         # session standardisation uses the EEG of every trial (transductive)
            mixture("combo", all_encs, sess=True)
        mixture("combo_nosess", all_encs, sess=False)

        if a.explicit and abs(frac - a.explicit_frac) < 1e-9:
            # explicit test predictions of the final models: band / 50 Hz / single-trial metrics
            res["explicit"] = {}
            Yt = [Y[i] for i in test]
            for nm in [m for m in ("base", "enc", "combo", "combo_nosess") if m in store]:
                rows, A_, c_, names = store[nm]
                P = []
                for i in test:
                    X = main_x(E[i])[0][rows].astype(np.float64)
                    if names:
                        mm, dd = tf_for(names, ses[i])
                        X = (X - mm[:, None]) / dd[:, None]
                    P.append((X.T @ A_ + c_).T.astype(np.float32))
                met = per_contact_metrics(P, Yt, FS, bands=True)
                met["r_no50"], met["r_induced"] = r_without_line(P, Yt), r_single_trial(P, Yt)
                if events is not None:
                    met["r_event"] = r_event(P, Yt, test, events)
                Pc = np.concatenate(P, 1)
                res["explicit"][nm] = {"frac": frac, "mean": {k: float(np.nanmean(v)) for k, v in met.items()},
                                       "r": [float(x) for x in met["r"]],
                                       "r_induced": [float(x) for x in met["r_induced"]],
                                       "n_const": int((Pc.max(1) == Pc.min(1)).sum())}   # constant -> r = 0
            log("    explicit: " + " ".join(f"{k} r={v['mean']['r']:.4f} (stats {out[k]['r']:.4f}) "
                                          f"r_ind={v['mean']['r_induced']:.4f}"
                                          for k, v in res["explicit"].items()))
        mins = sum(E[i].shape[1] for i in range(n_c)) / FS / 60
        res["fracs"][f"{frac:g}"] = {"n_calib": n_c, "minutes": mins, "models": out, "choice": extra,
                                     "calib_sessions": sorted({ses[i] for i in range(n_c)}, key=order.index)}
        log(f"    f={frac:g} ({mins:.1f} min, {time.time() - t0:.0f}s) " +
            " ".join(f"{k}={v['r']:.3f}" for k, v in out.items()))
    return res


# ----------------------------------------------------------------------------- summary
def summarize(results, out_dir, log):
    from scipy.stats import wilcoxon
    lines, summ = [], {}
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r["fracs"][f] for r in results if f in r["fracs"]]
        models = [m for m in rows[0]["models"] if all(m in r["models"] for r in rows)]
        M = {m: np.array([r["models"][m]["r"] for r in rows]) for m in models}
        R2 = {m: np.array([r["models"][m]["r2"] for r in rows]) for m in models}
        mins = np.mean([r["minutes"] for r in rows])
        lines += [f"\n### calib fraction {f}  ({len(rows)} subjects, mean {mins:.1f} min)",
                  "| model | mean r | vs enc | wins vs enc | p vs enc | vs base | R² (calibrated) | K-fold choices |",
                  "|---|---|---|---|---|---|---|---|"]
        summ[f] = {}
        for m in models:
            d = M[m] - M["enc"]
            try:
                p = float(wilcoxon(M[m], M["enc"]).pvalue) if m != "enc" and np.any(d != 0) else float("nan")
            except ValueError:
                p = float("nan")
            ch = [r["choice"].get(m) for r in rows if m in r["choice"]]
            ch = [tuple(x) if isinstance(x, list) else x for x in ch]
            if ch and m != "enc_bands":
                vals, cnt = np.unique([str(x) for x in ch], return_counts=True)
                chs = ", ".join(f"{v}×{c}" for v, c in zip(vals, cnt))
            else:
                chs = ""
            summ[f][m] = {"mean": float(M[m].mean()), "diff_enc": float(d.mean()), "wins_enc": int((d > 0).sum()),
                          "p_enc": p, "diff_base": float((M[m] - M["base"]).mean()), "r2": float(R2[m].mean())}
            lines.append(f"| {m} | {M[m].mean():.4f} | {d.mean():+.4f} | {(d > 0).sum()}/{len(d)} | {p:.4f} | "
                         f"{(M[m] - M['base']).mean():+.4f} | {R2[m].mean():.3f} | {chs} |")
    md = "\n".join(lines)
    log(md)
    with open(os.path.join(out_dir, "readout.md"), "w", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "readout.json"), "w") as fh:
        json.dump(summ, fh, indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--run", required=True, help="output dir of s2i.shared (enc_<sub>.pt)")
    ap.add_argument("--seed_run", default=None, help="second run for the seed ensemble (enc_<sub>.pt)")
    ap.add_argument("--seed_runs", nargs="*", default=[], help="further runs (encoders) for the ensemble")
    ap.add_argument("--explicit", action="store_true",
                    help="also predict the test trials explicitly with base / enc / combo (all metrics)")
    ap.add_argument("--explicit_frac", type=float, default=0.2)
    ap.add_argument("--strict", action="store_true",
                    help="re-normalise EEG and target with calibration-trial statistics only (needs <sub>_normstats.pt "
                         "from preprocess --stats_only); no session standardisation; requires a single --fracs value")
    ap.add_argument("--stats_dir", default=None, help="folder with <sub>_normstats.pt (default: --data)")
    ap.add_argument("--untrained", action="store_true",
                    help="replace every encoder by its Morlet initialisation (fixed filter bank, no meta-learning)")
    ap.add_argument("--only_combo", action="store_true",
                    help="skip the Phase-4 variants already rejected; compute enc, base, ensembles and combos")
    ap.add_argument("--out_name", default=None, help="output sub-folder (default: readout / readout_combo)")
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.05, 0.1, 0.2])
    ap.add_argument("--subjects", nargs="*", default=None)
    ap.add_argument("--no_bilin", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    if a.strict:
        assert len(a.fracs) == 1, "--strict needs exactly one calibration fraction (the statistics depend on it)"
    out = os.path.join(a.run, a.out_name or ("readout_strict" if a.strict else
                                             "readout_combo" if a.only_combo else "readout"))
    os.makedirs(out, exist_ok=True)
    log_f = open(os.path.join(out, "readout.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    torch.set_grad_enabled(False)
    global _DEV
    _DEV = dev
    seed_runs = ([a.seed_run] if a.seed_run else []) + list(a.seed_runs)
    log(f"config: {vars(a)}")
    results = []
    for ep in sorted(glob.glob(os.path.join(a.run, "enc_sub-*.pt"))):
        sid = os.path.basename(ep)[4:-3]
        if a.subjects and sid not in a.subjects:
            continue
        fp = os.path.join(out, f"readout_{sid}.json")
        sps = [sp for sp in (os.path.join(r_, f"enc_{sid}.pt") for r_ in seed_runs) if os.path.exists(sp)]
        if not os.path.exists(os.path.join(a.data, f"{sid}.pt")):
            log(f"[{sid}] no data file in {a.data} (excluded at this level), skip")
            continue
        if os.path.exists(fp) and not a.overwrite:
            done = json.load(open(fp))
            # resume: keep a finished subject only if it used the same number of encoders as now
            if done.get("n_encoders", 1 + len(sps)) == 1 + len(sps):
                results.append(done)
                log(f"[{sid}] already done, loaded")
                continue
            log(f"[{sid}] done with {done['n_encoders']} encoders, now {1 + len(sps)}: recomputing")
        t0 = time.time()
        enc = load_encoder(ep, dev)
        seed_encs = [load_encoder(sp, dev) for sp in sps]
        if a.untrained:
            enc = TemporalEncoder(**enc.cfg).to(dev).eval()
            seed_encs = [TemporalEncoder(**se.cfg).to(dev).eval() for se in seed_encs]
        E, Y, ses, events = load_subject(os.path.join(a.data, f"{sid}.pt"))
        if a.strict:
            st = torch.load(os.path.join(a.stats_dir or a.data, f"{sid}_normstats.pt"), weights_only=False)
            assert len(st["trial_bounds"]) == len(E) and list(st["trial_session"]) == ses, "stats / data mismatch"
            E, Y = strict_renorm(E, Y, ses, st, max(3, int(round(len(E) * a.fracs[0]))))
        log(f"[{sid}] {len(E)} trials, {E[0].shape[0]} scalp, {Y[0].shape[0]} targets, "
            f"{len(set(ses))} sessions, extra encoders: {len(seed_encs)}")
        r = analyze(E, Y, ses, enc, seed_encs, a, dev, log, events)
        r["subject"] = sid
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
    if results:
        summarize(results, out, log)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/review13.py
"""
Phase 13 - third external review: common mode in VOLTS, and quality-control (QC) statistics.

  volts   The Phase-11 common-mode analysis (review11 refcheck) worked on data normalised per channel and per
          session (median / MAD), where a channel mean is not the physical common mode. Here every signal is
          first returned to filtered volts (x * MAD_session + median_session, statistics from
          <sub>_normstats.pt), the common modes and the "monopolar - common mode" targets are formed in volts,
          and only then is every signal normalised with CALIBRATION-trial statistics (strict protocol).
            iEEG common mode  = mean of all retained monopolar contacts (volts, recording reference)
            scalp common mode = mean of all retained scalp channels (volts, original reference)
          Same lagged ridge, same clean split as review11. iEEG samples clipped at |z| = 10 during preprocessing
          stay clipped (their share is reported by `qc`).
  qc      per patient: trials kept / rejected by the scalp-artefact rule, share of test samples at the iEEG
          clipping bound (bipolar and monopolar), and the raw-volt common-mode diagnostics saved by preprocessing
          (correlation of the raw iEEG mean with each raw scalp channel, first-eigenvalue share of the monopolar
          correlation matrix), all computed on the whole session before any re-referencing.

Usage
-----
python -m s2i.review13 volts --data DATA_DIR --out RUNS_DIR/review13
python -m s2i.review13 qc    --data DATA_DIR --out RUNS_DIR/review13
"""
import argparse
import glob
import json
import os
import sys

import numpy as np
import torch

FS = 200
RNG = np.random.default_rng(0)


def boot_ci(d, n=10000):
    d = np.asarray(d, float)
    m = np.array([d[RNG.integers(0, len(d), len(d))].mean() for _ in range(n)])
    return np.percentile(m, 2.5), np.percentile(m, 97.5)


def wil(a, b):
    from scipy.stats import wilcoxon
    try:
        return float(wilcoxon(a, b).pvalue)
    except ValueError:
        return float("nan")


PATTERN = "sub-[0-9][0-9].pt"                         # ds004752; --pattern for other datasets (e.g. sub-P*.pt)


def subjects(data):
    return sorted(p for p in glob.glob(os.path.join(data, PATTERN)) if not p.endswith("_normstats.pt"))


def load_volts(path):
    """-> trial bounds, trial sessions, {key: volts (C, T)} for eeg_orig / ieeg_mono / ieeg_bip, data dict."""
    d = torch.load(path, weights_only=False)
    sp = path[:-3] + "_normstats.pt"
    st = torch.load(sp, weights_only=False) if os.path.exists(sp) else d
    ns, ses = st["norm_stats"], list(st["trial_session"])
    bounds = d["trial_bounds"].numpy()
    assert len(ses) == len(bounds), "normstats / data mismatch"
    out = {}
    for key in ("eeg_orig", "ieeg_mono", "ieeg_bip"):
        x = d[key].half().float().numpy().astype(np.float64)
        v = np.empty_like(x)
        for (s0, s1), s in zip(bounds, ses):
            med = np.asarray(ns[key][s]["med"], float)[:, None]
            mad = np.asarray(ns[key][s]["mad"], float)[:, None]
            v[:, s0:s1] = x[:, s0:s1] * mad + med
        out[key] = v
    return bounds, ses, out, d


def calib_norm(x, bounds, n_cal):
    cal = np.concatenate([x[:, s0:s1] for s0, s1 in bounds[:n_cal]], 1)
    med = np.median(cal, axis=1, keepdims=True)
    mad = np.median(np.abs(cal - med), axis=1, keepdims=True) * 1.4826
    return ((x - med) / np.where(mad < 1e-30, 1.0, mad)).astype(np.float32)


def volts(a, log):
    from s2i.protocol import ALPHAS, LAG9, run_variant, split
    from s2i.metrics import per_contact_metrics
    os.makedirs(a.out, exist_ok=True)
    res = {}
    for p in subjects(a.data):
        sid = os.path.basename(p)[:-3]
        fp = os.path.join(a.out, f"volts_{sid}.json")
        if os.path.exists(fp):
            res[sid] = json.load(open(fp))
            continue
        bounds, ses, V, _ = load_volts(p)
        if len(bounds) < 20:
            continue
        lens = [int(s1 - s0) for s0, s1 in bounds]
        train, test, folds = split("clean", lens, np.random.default_rng(0))
        n_cal = len(train)
        e, mono, bip = V["eeg_orig"], V["ieeg_mono"], V["ieeg_bip"]
        cm_i, cm_e = mono.mean(0, keepdims=True), e.mean(0, keepdims=True)
        te = np.concatenate([np.arange(bounds[i][0], bounds[i][1]) for i, _, _ in test])
        r_cm = float(np.corrcoef(cm_i[0, te], cm_e[0, te])[0, 1])
        share = float(np.nanmean([np.corrcoef(mono[k, te], cm_i[0, te])[0, 1] ** 2 for k in range(mono.shape[0])]))
        inputs = {"orig": calib_norm(e, bounds, n_cal), "car": calib_norm(e - cm_e, bounds, n_cal)}
        targets = {"mono": calib_norm(mono, bounds, n_cal), "mono_minus_cm": calib_norm(mono - cm_i, bounds, n_cal),
                   "bip": calib_norm(bip, bounds, n_cal), "cm": calib_norm(cm_i, bounds, n_cal)}
        out = {"subject": sid, "r_scalp_cm_vs_ieeg_cm": r_cm, "ieeg_variance_share_of_cm": share, "pred": {}}
        for ref, X in inputs.items():
            E = [X[:, s0:s1] for s0, s1 in bounds]
            for tg, Yall in targets.items():
                Y = [Yall[:, s0:s1] for s0, s1 in bounds]
                P, T, _ = run_variant(LAG9, ALPHAS, E, Y, train, test, folds)
                out["pred"][f"{tg}|{ref}"] = float(np.nanmean(per_contact_metrics(P, T, FS, bands=False)["r"]))
        json.dump(out, open(fp, "w"))
        res[sid] = out
        log(f"[{sid}] volts: scalp-cm vs iEEG-cm r={r_cm:+.3f}, cm share={share:.2f} | " +
            " ".join(f"{k}={v:+.3f}" for k, v in out["pred"].items()))
    subs = sorted(res)
    keys = list(res[subs[0]]["pred"])
    M = {k: np.array([res[s]["pred"][k] for s in subs]) for k in keys}
    old = {}
    for s in subs:                                    # Phase-11 values (normalised domain), if present
        f = os.path.join(a.old, f"refcheck_{s}.json") if a.old else ""
        if f and os.path.exists(f):
            old[s] = json.load(open(f))
    lab = {"mono": "monopolar", "mono_minus_cm": "monopolar − iEEG common mode", "bip": "bipolar",
           "cm": "iEEG common mode itself"}
    L = ["# Common mode in volts (strict normalisation afterwards; lagged ridge, clean split)", "",
         "| target \\ scalp reference | original (volts) | CAR (volts) | original, Phase 11 (normalised domain) |",
         "|---|---|---|---|"]
    for tg in ("mono", "mono_minus_cm", "bip", "cm"):
        o = (np.mean([old[s]["pred"][tg + "|orig"] for s in old]) if len(old) == len(subs) else np.nan)
        L.append(f"| {lab[tg]} | {M[tg + '|orig'].mean():.3f} | {M[tg + '|car'].mean():.3f} | {o:.3f} |")
    d = M["mono|orig"] - M["mono_minus_cm|orig"]
    lo, hi = boot_ci(d)
    L += ["", f"- removing the iEEG common mode in volts (original scalp): {-d.mean():+.3f} [{-hi:+.3f}, {-lo:+.3f}], "
          f"lower in {int((d > 0).sum())}/{len(subs)} patients, p = {wil(M['mono|orig'], M['mono_minus_cm|orig']):.5f}"]
    x = M["mono|orig"] - M["mono|car"] - M["bip|orig"] + M["bip|car"]
    lo, hi = boot_ci(x)
    L += [f"- interaction (mono − bip) × (orig − CAR), volts: {x.mean():+.3f} [{lo:+.3f}, {hi:+.3f}], "
          f"positive in {int((x > 0).sum())}/{len(subs)}, p = {wil(x, np.zeros_like(x)):.5f}"]
    if len(old) == len(subs):
        dv = np.array([res[s]["pred"]["mono_minus_cm|orig"] - old[s]["pred"]["mono_minus_cm|orig"] for s in subs])
        lo, hi = boot_ci(dv)
        L += [f"- monopolar − common mode, volts minus normalised-domain: {dv.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]"]
    L += ["", "| patient | scalp-cm vs iEEG-cm r (volts) | cm variance share (volts) | mono, orig | mono − cm, orig | "
          "bip, CAR | cm, orig | cm, CAR |", "|---|---|---|---|---|---|---|---|"]
    for s in subs:
        q, p_ = res[s], res[s]["pred"]
        L.append(f"| {s} | {q['r_scalp_cm_vs_ieeg_cm']:+.2f} | {q['ieeg_variance_share_of_cm']:.2f} | "
                 f"{p_['mono|orig']:.3f} | {p_['mono_minus_cm|orig']:.3f} | {p_['bip|car']:.3f} | "
                 f"{p_['cm|orig']:.3f} | {p_['cm|car']:.3f} |")
    out = "\n".join(L)
    log(out)
    open(os.path.join(a.out, "volts.md"), "w", encoding="utf-8").write(out + "\n")


def qc(a, log):
    from s2i.protocol import split
    rows = []
    for p in subjects(a.data):
        d = torch.load(p, weights_only=False)
        b = d["trial_bounds"].numpy()
        lens = [int(s1 - s0) for s0, s1 in b]
        _, test, _ = split("clean", lens, np.random.default_rng(0))
        te = np.concatenate([np.arange(b[i][0], b[i][1]) for i, _, _ in test])
        clip = float(d.get("norm_clip_z", 10.0))
        frac = {}
        for k in ("ieeg_bip", "ieeg_mono"):
            x = d[k].half().float().numpy()[:, te]
            frac[k] = float((np.abs(x) >= clip * 0.999).mean())
        dg = d.get("diagnostics", {}) or {}
        rows.append((os.path.basename(p)[:-3], len(b), int(d.get("n_rejected_trials", -1)), frac["ieeg_bip"],
                     frac["ieeg_mono"], dg.get("mean_abs_corr_iEEGmean_vs_EEG", np.nan),
                     dg.get("max_abs_corr_iEEGmean_vs_EEG", np.nan), dg.get("mono_pc1_ratio", np.nan)))
    kept = sum(r[1] for r in rows)
    rej = sum(max(r[2], 0) for r in rows)
    L = ["# Quality control (thresholds computed from whole-session median / MAD)", "",
         f"- trials kept {kept}, rejected by the scalp-artefact rule (|z| > 15, CAR EEG) {rej} "
         f"({100 * rej / max(kept + rej, 1):.2f} %); trials marked as artefacts in the dataset are excluded before this",
         f"- test samples at the iEEG clipping bound (|z| = 10): bipolar median "
         f"{100 * np.median([r[3] for r in rows]):.3f} % (max {100 * max(r[3] for r in rows):.3f} %), monopolar median "
         f"{100 * np.median([r[4] for r in rows]):.3f} % (max {100 * max(r[4] for r in rows):.3f} %)", "",
         "| patient | trials kept | rejected | test bip clipped % | test mono clipped % | raw iEEG-mean vs scalp, mean |r| | "
         "max |r| | mono PC1 share |", "|---|---|---|---|---|---|---|---|"]
    L += [f"| {s} | {k} | {r} | {100 * fb:.3f} | {100 * fm:.3f} | {m1:.2f} | {m2:.2f} | {pc:.2f} |"
          for s, k, r, fb, fm, m1, m2, pc in rows]
    out = "\n".join(L)
    log(out)
    os.makedirs(a.out, exist_ok=True)
    open(os.path.join(a.out, "qc.md"), "w", encoding="utf-8").write(out + "\n")


def main():
    global PATTERN
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=("volts", "qc"))
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--old", default=None, help="folder with Phase-11 refcheck_<sub>.json (for the comparison column)")
    ap.add_argument("--pattern", default=PATTERN, help="subject file pattern inside --data")
    a = ap.parse_args()
    PATTERN = a.pattern
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    {"volts": volts, "qc": qc}[a.cmd](a, print)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/scales17.py
"""
Phase 17 - external validation on ScaleOLDNEW, exactly as in paper/external_validation_protocol.md (v1.6, locked).

Commands
  build   Phase-14 output (proc/<patient>_Block<k>.npz: notch 50/100 Hz, 0.5-70 Hz, 200 Hz, microvolts) + the raw
          ADES files and spreadsheets on Drive -> one data folder per evaluation level, in the s2i format read by
          s2i.readout --strict and s2i.review13 volts:
            primary      all windows, no automatic artefact rejection, no target clipping
            secondary    windows with |z| > 15 (CAR scalp EEG) removed, targets clipped at |z| = 10, median / MAD of
                         the whole block
            sensitivity  the same thresholds, median / MAD of the first 20 % of the windows
          plus the execution log (p17_build.md / p17_build.json): channel rules and the Table-1 consistency check,
          non-finite counts (raw data), recording gaps, windows, split, inclusion, target validity, markers.
  runs    copy each final encoder (s2i.shared --final -> <variant>/enc_final.pt) to one file per patient
          (<runs>/<variant>/enc_sub-Pxx.pt), so that s2i.readout runs on ScaleOLDNEW unchanged
  report  H1 / H2 (paired two-sided Wilcoxon, Holm over H1-H2 only, median direction, patient bootstrap CI of the
          mean), descriptive analyses, constant-prediction counts, common mode by recording setup
          -> p17_report.md / p17_report.json

Rules implemented here (protocol sections in brackets):
  scalp channel [4]   EEG type in the header; name (after the dataset's lookup table) is an MNE 10-05 position; not
                      A1/A2/M1/M2; not bad in any block; non-zero SD in every block; no non-finite raw sample
  target [4, 7]       pair of the BipolarChannels sheet; both contacts recorded in every block, never bad, no
                      non-finite raw sample; bipolar = A - B (microvolts); non-zero variance in calibration and test
  windows [5]         8 s, consecutive, inside each block (file) only, 3.4 s from the block edges; a window
                      overlapping a recording gap (>= 50 ms where all kept scalp channels, or all target contacts, are
                      exactly 0 or non-finite in the RAW data) or its 3.4 s margins is dropped; the last piece of a
                      block is kept if >= 2 s
  split [5]           windows ordered by block and time; n_cal = max(3, round(0.2 N)); 1-window gap; rest = test;
                      patient included only if n_cal >= 5, >= 3 scalp channels and >= 3 valid targets
  CAR [4]             mean of the kept scalp channels subtracted at every sample (microvolts), before normalisation
Storage: every signal is robust-normalised with the median / MAD of ALL kept samples of the patient and stored as
float16 together with these statistics (norm_stats, one pseudo-session "all"); s2i.readout --strict undoes this and
re-normalises with the calibration windows only, exactly as for ds004752.

Usage
-----
python -m s2i.scales17 build  --sc_root SC_ROOT --out DATA17
python -m s2i.scales17 runs   --enc_root ENC17 --data DATA17/primary --runs RUNS17
python -m s2i.scales17 report --data DATA17 --runs RUNS17
"""
import argparse
import glob
import json
import os
import re
import shutil
import sys
import time

import numpy as np

FS = 200
WIN, MIN_REM, EDGE = 8 * FS, 2 * FS, 680              # 8 s windows, >= 2 s remainder, 3.4 s margins (samples)
GAP_SEC = 0.05
CAL_FRAC, MIN_CAL, MIN_SCALP, MIN_TGT = 0.2, 5, 3, 3
REJECT_Z, CLIP_Z = 15.0, 10.0
LEVELS = ("primary", "secondary", "sensitivity")
LOOKUP = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8", "FB1": "FT9", "FB2": "FT10", "TP1": "TP9", "TP2": "TP10"}
EXCLUDED_NAMES = {"A1", "A2", "M1", "M2"}
# Table 1 of Barborica et al. 2023: (patient number, ID, recording setup, number of scalp electrodes).
# Setup from the Methods: patients 1-8 two 128-channel amplifiers, patients 9-12 one 256-channel amplifier.
# Scalp counts transcribed from the article's HTML table; check against the PDF.
PATIENTS = [(1, 89, "two amplifiers", 30), (2, 90, "two amplifiers", 30), (3, 92, "two amplifiers", 30),
            (4, 96, "two amplifiers", 38), (5, 97, "two amplifiers", 35), (6, 98, "two amplifiers", 38),
            (7, 99, "two amplifiers", 38), (8, 101, "two amplifiers", 40), (9, 102, "one amplifier", 40),
            (10, 104, "one amplifier", 40), (11, 105, "one amplifier", 37), (12, 107, "one amplifier", 40)]
CONTACT = re.compile(r"^[A-Z]{1,2}'?\d{1,2}$")


def log(msg):
    print(f"{time.strftime('%H:%M:%S')} | {msg}", flush=True)


def n_cal_of(n):
    return max(3, int(round(n * CAL_FRAC)))


def robust(x):
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) * 1.4826
    return med, np.where(mad < 1e-30, 1.0, mad)


# ----------------------------------------------------------------------------- inputs
def template_1005():
    import warnings
    import mne
    with warnings.catch_warnings():                   # renamed colin27_1005 from MNE 1.14 (same positions)
        warnings.simplefilter("ignore", FutureWarning)
        try:
            mont = mne.channels.make_standard_montage("standard_1005")
        except ValueError:
            mont = mne.channels.make_standard_montage("colin27_1005")
    return {n.upper(): n for n in mont.ch_names}


def scalp_name(name, t1005):
    """-> (10-05 name or None, reason)."""
    up = LOOKUP.get(name.strip().upper(), name.strip().upper()).upper()
    if up in EXCLUDED_NAMES:
        return None, "A1/A2/M1/M2"
    if up not in t1005:
        return None, "not a 10-05 position"
    return t1005[up], ""


def load_block(path):
    z = np.load(path, allow_pickle=True)
    out = {k: z[k] for k in ("eeg", "seeg")}
    for k in ("eeg_names", "seeg_names", "bad"):
        out[k] = [str(x).strip() for x in z[k]]
    out["markers"] = [(str(m[0]).strip(), float(m[1])) for m in z["markers"]]
    out["fs_raw"], out["n_raw"] = float(z["fs_raw"]), int(z["n_raw"])
    return out


def read_header(path):
    fs, n, chans = None, None, []
    for ln in open(path, encoding="latin-1").read().splitlines():
        if ln.startswith("#") or "=" not in ln:
            continue
        k, v = [x.strip() for x in ln.split("=", 1)]
        if k == "samplingRate":
            fs = float(v)
        elif k == "numberOfSamples":
            n = int(v)
        else:
            chans.append((k, v))
    return fs, n, chans


def raw_scan(dat, ades, count_names, scalp, contacts, chunk=32768):
    """
    One pass over a raw block (float32, sample-major, microvolts): non-finite samples per channel of count_names,
    and recording gaps = runs of >= 50 ms in which all `scalp` channels, or all `contacts`, are exactly zero or
    non-finite. Gaps are returned as [start, stop) intervals at 200 Hz.
    """
    fs, n, chans = read_header(ades)
    col = {}
    for i, (c, _) in enumerate(chans):
        col.setdefault(c, i)
    assert os.path.getsize(dat) == 4 * len(chans) * n, f"size mismatch {dat}"
    X = np.memmap(dat, dtype="<f4", mode="r", shape=(n, len(chans)))
    cc = [col[c] for c in count_names]
    pos = {c: j for j, c in enumerate(count_names)}
    si, ti = [pos[c] for c in scalp], [pos[c] for c in contacts]
    nonfin = np.zeros(len(cc), np.int64)
    dead = np.zeros(n, bool)
    for k in range(0, n, chunk):
        x = np.asarray(X[k:k + chunk])[:, cc]
        bad = ~np.isfinite(x)
        nonfin += bad.sum(0)
        z = bad | (x == 0)
        dead[k:k + chunk] = (z[:, si].all(1) if si else False) | (z[:, ti].all(1) if ti else False)
    del X
    gaps = []
    d = np.diff(np.r_[0, dead.astype(np.int8), 0])
    for s0, s1 in zip(np.where(d == 1)[0], np.where(d == -1)[0]):
        if s1 - s0 >= int(np.ceil(GAP_SEC * fs)):
            gaps.append((int(np.floor(s0 * FS / fs)), int(np.ceil(s1 * FS / fs))))
    return dict(zip(count_names, nonfin.tolist())), gaps, fs


def windows_of(L, gaps):
    """Consecutive 8-s windows of one block -> (kept [(s0, s1)], number dropped for gaps, remainder kept / dropped)."""
    blocked = np.zeros(L, bool)
    blocked[:EDGE] = blocked[L - EDGE:] = True
    for g0, g1 in gaps:
        blocked[max(0, g0 - EDGE):min(L, g1 + EDGE)] = True
    kept, n_gap, rem = [], 0, "none"
    s, end = EDGE, L - EDGE
    while s < end:
        e = min(s + WIN, end)
        if e - s < WIN:
            if e - s < MIN_REM:
                rem = f"dropped ({(e - s) / FS:.1f} s)"
                break
            rem = f"kept ({(e - s) / FS:.1f} s)"
        if blocked[s:e].any():
            n_gap += 1
        else:
            kept.append((s, e))
        s = e
    return kept, n_gap, rem


# ----------------------------------------------------------------------------- one patient
def build_patient(a, num, pid, setup, t1005_n, t1005):
    sid = f"sub-P{num:02d}"
    raw_dir = os.path.join(a.sc_root, "raw", f"SEEG{pid}")
    files = sorted(glob.glob(os.path.join(a.sc_root, "proc", f"SEEG{pid}_Block*.npz")),
                   key=lambda p: int(re.search(r"Block(\d+)", p).group(1)))
    info = {"subject": sid, "patient": num, "id": pid, "setup": setup, "table1_scalp": t1005_n, "excluded": None}
    if not files:
        info["excluded"] = "no preprocessed block"
        return info, None
    blocks, nums = [], []
    for f in files:
        try:
            blocks.append(load_block(f))
            nums.append(int(re.search(r"Block(\d+)", f).group(1)))
        except Exception as e:                                         # unreadable block: skipped and reported
            info.setdefault("unreadable_blocks", []).append(f"{os.path.basename(f)}: {type(e).__name__}")
    info["blocks"] = nums
    bad_all = sorted({c for b in blocks for c in b["bad"]})
    info["bad_any_block"] = bad_all

    # ---- scalp channels (rules 1-5; non-finite from the raw scan below)
    cand = [c for c in blocks[0]["eeg_names"] if all(c in b["eeg_names"] for b in blocks)]
    reasons, keep, canon = {}, [], []
    for c in cand:
        nm, why = scalp_name(c, t1005)
        if nm is None:
            reasons[c] = why
        elif c in bad_all:
            reasons[c] = "bad"
        elif nm in canon:
            reasons[c] = f"duplicate of {nm}"
        elif any(np.std(b["eeg"][b["eeg_names"].index(c)].astype(np.float64)) == 0 for b in blocks):
            reasons[c] = "zero SD in a block"
        else:
            keep.append(c)
            canon.append(nm)

    # ---- targets: BipolarChannels pairs, contacts recorded in every block and never bad
    import pandas as pd
    xl = sorted(glob.glob(os.path.join(raw_dir, "*ChannelData.xlsx")))
    sheet = pd.read_excel(xl[0], sheet_name="BipolarChannels")
    recorded = set.intersection(*[set(b["seeg_names"]) for b in blocks])
    pairs, t_reason = [], {}
    for _, row in sheet.iterrows():
        nm = str(row["name"]).strip()
        parts = nm.split("-")
        if len(parts) != 2:
            t_reason[nm] = "name not A-B"
            continue
        p0, p1 = parts[0].strip(), parts[1].strip()
        if p0 not in recorded or p1 not in recorded:
            t_reason[nm] = "contact not recorded in every block"
        elif p0 in bad_all or p1 in bad_all:
            t_reason[nm] = "contact bad"
        else:
            pairs.append({"name": nm, "a": p0, "b": p1, "mni": [float(row[k]) for k in ("x_mni", "y_mni", "z_mni")],
                          "structure": str(row.get("structure", ""))})

    # ---- raw scan: non-finite counts and recording gaps (repeated if the non-finite rule changes the sets)
    seeg_all = [c for c in blocks[0]["seeg_names"] if c in recorded]
    count_names = cand + seeg_all
    nonfin, gaps, fs_raw = {c: 0 for c in count_names}, {}, {}
    for it in range(3):
        contacts = sorted({p["a"] for p in pairs} | {p["b"] for p in pairs})
        nonfin = {c: 0 for c in count_names}
        for k, b in zip(nums, blocks):
            base = os.path.join(raw_dir, f"Block{k}", f"SEEG{pid}Block{k}")
            nf, gaps[k], fs_raw[k] = raw_scan(base + ".dat", base + ".ades", count_names, keep, contacts)
            for c, v in nf.items():
                nonfin[c] += v
        bad_scalp = [c for c in keep if nonfin[c] > 0]
        bad_ct = {c for c in contacts if nonfin[c] > 0}
        if not bad_scalp and not bad_ct:
            break
        for c in bad_scalp:
            reasons[c] = "non-finite raw sample"
            canon.pop(keep.index(c))
            keep.remove(c)
        for p in [p for p in pairs if p["a"] in bad_ct or p["b"] in bad_ct]:
            t_reason[p["name"]] = "contact non-finite"
            pairs.remove(p)
    info.update(scalp_candidates=len(cand), scalp_kept=canon, n_scalp=len(keep), scalp_excluded=reasons,
                nonfinite={c: v for c, v in nonfin.items() if v > 0}, nonfinite_channels_checked=len(count_names),
                n_pairs_sheet=int(len(sheet)), pairs_excluded=t_reason, fs_raw=fs_raw,
                gaps={k: v for k, v in gaps.items()})
    contacts = sorted({p["a"] for p in pairs} | {p["b"] for p in pairs})

    # ---- signals per block (microvolts) and windows
    S, W, wl = {}, [], {}
    for k, b in zip(nums, blocks):
        e = np.stack([b["eeg"][b["eeg_names"].index(c)] for c in keep]).astype(np.float64)
        m = np.stack([b["seeg"][b["seeg_names"].index(c)] for c in contacts]).astype(np.float64) if contacts \
            else np.zeros((0, e.shape[1]))
        L = min(e.shape[1], m.shape[1])
        e, m = e[:, :L], m[:, :L]
        ci = {c: i for i, c in enumerate(contacts)}
        bip = np.stack([m[ci[p["a"]]] - m[ci[p["b"]]] for p in pairs]) if pairs else np.zeros((0, L))
        S[k] = {"orig": e, "car": e - e.mean(0, keepdims=True), "mono": m, "bip": bip}
        kw, n_gap, rem = windows_of(L, gaps[k])
        stim = [(c, t) for c, t in b["markers"] if c.isdigit()]
        codes = {}
        for c, _ in stim:
            codes[c] = codes.get(c, 0) + 1
        wl[k] = {"minutes": L / FS / 60, "windows": len(kw), "dropped_gap": n_gap, "remainder": rem,
                 "gaps": len(gaps[k]), "gap_seconds": sum(g1 - g0 for g0, g1 in gaps[k]) / FS, "stim_markers": codes,
                 "other_markers": sorted({c for c, _ in b["markers"] if not c.isdigit()})}
        onsets = np.array([int(round(t * FS)) for _, t in stim], int)
        for s0, s1 in kw:
            W.append({"block": k, "s0": s0, "s1": s1, "onsets": (onsets[(onsets >= s0) & (onsets < s1)] - s0).tolist()})
    info["block_windows"] = wl
    info["n_windows"] = len(W)
    if len(keep) < MIN_SCALP:
        info["excluded"] = f"{len(keep)} scalp channels (< {MIN_SCALP})"
    meta = dict(sid=sid, pid=pid, setup=setup, canon=canon, pairs=pairs, contacts=contacts)
    return info, (S, W, meta)


def level_windows(level, S, W):
    """-> kept windows and their signals for one evaluation level (rejection / clipping as in the protocol)."""
    sig = [{k: S[w["block"]][k][:, w["s0"]:w["s1"]] for k in ("orig", "car", "mono", "bip")} for w in W]
    if level == "primary":
        return W, sig, {}
    if level == "secondary":                          # whole-block median / MAD
        st = {k: {b: robust(S[b][k]) for b in S} for k in ("car", "bip")}
        get = lambda k, w: st[k][w["block"]]                                    # noqa: E731
    else:                                             # first 20 % of the windows, fixed for all later windows
        nc = n_cal_of(len(W))
        st = {k: robust(np.concatenate([s[k] for s in sig[:nc]], 1)) for k in ("car", "bip")}
        get = lambda k, w: st[k]                                                # noqa: E731
    kw, ks = [], []
    for w, s in zip(W, sig):
        med, mad = get("car", w)
        if np.abs((s["car"] - med) / mad).max() > REJECT_Z:
            continue
        s = dict(s)                                   # monopolar signals are used at the primary level only
        med, mad = get("bip", w)
        s["bip"] = np.clip(s["bip"], med - CLIP_Z * mad, med + CLIP_Z * mad)
        kw.append(w)
        ks.append(s)
    return kw, ks, {"rejected_windows": len(W) - len(kw)}


def write_level(a, level, info, data):
    import torch
    S, W, meta = data
    kw, ks, extra = level_windows(level, S, W)
    N = len(kw)
    nc = n_cal_of(N)
    out = {"n_windows": N, "n_cal": nc, "n_test": max(N - nc - 1, 0), **extra,
           "calib_minutes": sum(w["s1"] - w["s0"] for w in kw[:nc]) / FS / 60,
           "test_minutes": sum(w["s1"] - w["s0"] for w in kw[nc + 1:]) / FS / 60, "excluded": info["excluded"]}
    if out["excluded"] is None and nc < MIN_CAL:
        out["excluded"] = f"n_cal = {nc} (< {MIN_CAL}; N = {N})"
    valid, why = [], {}
    for j, p in enumerate(meta["pairs"]):
        vc = np.var(np.concatenate([s["bip"][j] for s in ks[:nc]])) if nc and ks else 0.0
        vt = np.var(np.concatenate([s["bip"][j] for s in ks[nc + 1:]])) if N > nc + 1 else 0.0
        if vc > 0 and vt > 0:
            valid.append(j)
        else:
            why[p["name"]] = "zero variance in calibration or test windows"
    out["n_valid_targets"], out["targets_zero_variance"] = len(valid), why
    if out["excluded"] is None and len(valid) < MIN_TGT:
        out["excluded"] = f"{len(valid)} valid targets (< {MIN_TGT})"
    if out["excluded"] is not None:
        return out
    pairs = [meta["pairs"][j] for j in valid]
    cts = sorted({p["a"] for p in pairs} | {p["b"] for p in pairs})
    ci = [meta["contacts"].index(c) for c in cts]
    arrays = {"eeg_orig": np.concatenate([s["orig"] for s in ks], 1),
              "eeg_car": np.concatenate([s["car"] for s in ks], 1),
              "ieeg_bip": np.concatenate([s["bip"][valid] for s in ks], 1)}
    if level == "primary":                            # monopolar only for the common-mode analysis (Section 8)
        arrays["ieeg_mono"] = np.concatenate([s["mono"][ci] for s in ks], 1)
    d = {"subject": meta["sid"], "dataset": "ScaleOLDNEW", "id": meta["pid"], "setup": meta["setup"], "fs": FS,
         "level": level, "eeg_names": meta["canon"], "contact_names": cts,
         "bip_names": [p["name"] for p in pairs], "bip_structure": [p["structure"] for p in pairs],
         "bip_mni": torch.tensor([p["mni"] for p in pairs], dtype=torch.float32),
         "bip_pairs": [(cts.index(p["a"]), cts.index(p["b"])) for p in pairs],
         "window_block": [w["block"] for w in kw], "window_start": [w["s0"] for w in kw],
         "stim_events": [(i, o) for i, w in enumerate(kw) for o in w["onsets"]],
         "norm_clip_z": float("inf") if level == "primary" else CLIP_Z, "norm_stats": {}}
    bounds, c = [], 0
    for w in kw:
        bounds.append((c, c + w["s1"] - w["s0"]))
        c += w["s1"] - w["s0"]
    d["trial_bounds"] = torch.tensor(bounds, dtype=torch.long)
    d["trial_session"] = ["all"] * N
    n_inf = 0
    for k, x in arrays.items():                       # storage normalisation (undone by readout --strict)
        med, mad = robust(x)
        z = (x - med) / mad
        n_inf += int((np.abs(z) > 6e4).sum())
        d[k] = torch.from_numpy(np.clip(z, -6e4, 6e4).astype(np.float16))
        d["norm_stats"][k] = {"all": {"med": med[:, 0].tolist(), "mad": mad[:, 0].tolist()}}
    out["float16_overflow_samples"] = n_inf
    od = os.path.join(a.out, level)
    os.makedirs(od, exist_ok=True)
    fp = os.path.join(od, f"{meta['sid']}.pt")
    torch.save(d, fp + ".tmp")
    os.replace(fp + ".tmp", fp)
    torch.save({k: d[k] for k in ("subject", "norm_stats", "norm_clip_z", "trial_bounds", "trial_session")},
               os.path.join(od, f"{meta['sid']}_normstats.pt"))
    out["stim_events_in_windows"] = len(d["stim_events"])
    return out


def filter_record():
    """Filter / resampling configuration of the installed MNE (the Phase-14 preprocessing used the same functions)."""
    import inspect
    import mne
    from s2i.data.preprocess_ds004752 import BAND, NOTCH
    h = mne.filter.create_filter(None, 4096.0, BAND[0], BAND[1], verbose=False)
    sig = lambda f: {k: str(v.default) for k, v in inspect.signature(f).parameters.items()      # noqa: E731
                     if v.default is not inspect.Parameter.empty}
    return {"mne": mne.__version__, "notch_hz": list(NOTCH), "band_hz": list(BAND),
            "bandpass_length_at_4096_hz_s": len(h) / 4096.0,
            "notch_filter_defaults": sig(mne.io.BaseRaw.notch_filter), "filter_defaults": sig(mne.io.BaseRaw.filter),
            "resample_defaults": sig(mne.io.BaseRaw.resample)}


def build(a):
    os.makedirs(a.out, exist_ok=True)
    t1005 = template_1005()
    jp = os.path.join(a.out, "p17_build.json")
    res = json.load(open(jp)) if os.path.exists(jp) else {"patients": {}}
    res["filters"] = filter_record()
    lt = os.path.join(a.sc_root, "raw", "ElectrodeLabelsLookupTable.txt")
    res["lookup_table_file"] = open(lt, encoding="latin-1").read() if os.path.exists(lt) else "not found"
    res["rules"] = {"window_s": WIN / FS, "edge_margin_s": EDGE / FS, "min_remainder_s": MIN_REM / FS,
                    "gap_min_s": GAP_SEC, "cal_frac": CAL_FRAC, "min_cal": MIN_CAL, "min_scalp": MIN_SCALP,
                    "min_targets": MIN_TGT, "reject_z": REJECT_Z, "clip_z": CLIP_Z, "lookup": LOOKUP}
    for num, pid, setup, t1 in PATIENTS:
        sid = f"sub-P{num:02d}"
        if a.patients and sid not in a.patients and str(pid) not in a.patients:
            continue
        if sid in res["patients"] and all(lv in res["patients"][sid].get("levels", {}) for lv in LEVELS):
            log(f"[{sid}] already built")
            continue
        t0 = time.time()
        info, data = build_patient(a, num, pid, setup, t1, t1005)
        info["levels"] = {}
        if data is not None:
            for lv in LEVELS:
                info["levels"][lv] = write_level(a, lv, info, data)
        del data
        res["patients"][sid] = info
        json.dump(res, open(jp + ".tmp", "w"), indent=1, default=str)
        os.replace(jp + ".tmp", jp)
        lv = info["levels"].get("primary", {})
        log(f"[{sid}] ID {pid}: {info.get('n_scalp', 0)} scalp (Table 1: {t1}), "
            f"{lv.get('n_valid_targets', 0)} targets, {lv.get('n_windows', 0)} windows "
            f"(calib {lv.get('n_cal', 0)}, test {lv.get('n_test', 0)}), excluded: {lv.get('excluded', info['excluded'])} | "
            f"{(time.time() - t0) / 60:.1f} min")
    write_build_md(a, res)


def write_build_md(a, res):
    P = res["patients"]
    f = res["filters"]
    L = ["# Phase 17 execution log: data (protocol v1.6, Sections 4, 5, 9)", "",
         f"- MNE {f['mne']}; notch {f['notch_hz']} Hz, band-pass {f['band_hz']} Hz (band-pass FIR length at 4096 Hz: "
         f"{f['bandpass_length_at_4096_hz_s']:.2f} s); defaults: notch {f['notch_filter_defaults']}, "
         f"filter {f['filter_defaults']}, resample {f['resample_defaults']}",
         f"- rules: {res['rules']}", "", "## Scalp channels (consistency check against Table 1; not used to change rules)", "",
         "| patient | ID | setup | blocks | candidates (EEG in every block) | kept | Table 1 | excluded (reason: n) |",
         "|---|---|---|---|---|---|---|---|"]
    for sid, i in P.items():
        rs = {}
        for _, why in i.get("scalp_excluded", {}).items():
            rs[why] = rs.get(why, 0) + 1
        L.append(f"| {sid} | {i['id']} | {i['setup']} | {len(i.get('blocks', []))} | {i.get('scalp_candidates', 0)} | "
                 f"{i.get('n_scalp', 0)} | {i['table1_scalp']} | " + ", ".join(f"{k}: {v}" for k, v in rs.items()) + " |")
    L += ["", "Kept scalp channels per patient:", ""]
    L += [f"- {sid}: {', '.join(i.get('scalp_kept', []))}" for sid, i in P.items()]
    L += ["", "## Non-finite raw samples (all scalp candidates and recorded contacts; channels with 0 not listed)", ""]
    for sid, i in P.items():
        nf = i.get("nonfinite", {})
        L.append(f"- {sid}: {i.get('nonfinite_channels_checked', 0)} channels checked; "
                 + (", ".join(f"{c}: {v}" for c, v in nf.items()) if nf else "0 non-finite samples"))
    L += ["", "## Targets", "", "| patient | pairs in sheet | after recording / bad / non-finite rules | valid (primary) | "
          "excluded (reason: n) |", "|---|---|---|---|---|"]
    for sid, i in P.items():
        rs = {}
        for _, why in i.get("pairs_excluded", {}).items():
            rs[why] = rs.get(why, 0) + 1
        lv = i["levels"].get("primary", {})
        for why in lv.get("targets_zero_variance", {}).values():
            rs[why] = rs.get(why, 0) + 1
        L.append(f"| {sid} | {i.get('n_pairs_sheet', 0)} | {i.get('n_pairs_sheet', 0) - len(i.get('pairs_excluded', {}))} | "
                 f"{lv.get('n_valid_targets', 0)} | " + ", ".join(f"{k}: {v}" for k, v in rs.items()) + " |")
    L += ["", "## Blocks, gaps, windows and markers", "",
          "| patient | block | min | raw Hz | gaps (n, s) | windows kept | dropped (gap) | last piece | stimulus markers | other markers |",
          "|---|---|---|---|---|---|---|---|---|---|"]
    for sid, i in P.items():
        for k, b in i.get("block_windows", {}).items():
            L.append(f"| {sid} | {k} | {b['minutes']:.1f} | {i['fs_raw'].get(k, i['fs_raw'].get(str(k), ''))} | "
                     f"{b['gaps']}, {b['gap_seconds']:.1f} | {b['windows']} | {b['dropped_gap']} | {b['remainder']} | "
                     f"{b['stim_markers']} | {len(b['other_markers'])} |")
    for lv in LEVELS:
        L += ["", f"## Split and inclusion: {lv}", "",
              "| patient | windows | rejected | calib | test | calib min | test min | valid targets | float16 clipped | excluded |",
              "|---|---|---|---|---|---|---|---|---|---|"]
        for sid, i in P.items():
            x = i["levels"].get(lv)
            if x is None:
                L.append(f"| {sid} | | | | | | | | | {i.get('excluded')} |")
                continue
            L.append(f"| {sid} | {x['n_windows']} | {x.get('rejected_windows', 0)} | {x['n_cal']} | {x['n_test']} | "
                     f"{x['calib_minutes']:.2f} | {x['test_minutes']:.1f} | {x['n_valid_targets']} | "
                     f"{x.get('float16_overflow_samples', '')} | {x['excluded'] or ''} |")
        inc = [s for s, i in P.items() if i["levels"].get(lv, {}).get("excluded") is None and i["levels"].get(lv)]
        L.append(f"\nincluded: {len(inc)} patients")
    out = "\n".join(L)
    open(os.path.join(a.out, "p17_build.md"), "w", encoding="utf-8").write(out + "\n")
    print(out)


# ----------------------------------------------------------------------------- runs
def runs(a):
    subs = sorted(os.path.basename(p)[:-3] for p in glob.glob(os.path.join(a.data, "sub-P*.pt"))
                  if not p.endswith("_normstats.pt"))
    for v in a.variants:
        src = os.path.join(a.enc_root, v, "enc_final.pt")
        assert os.path.exists(src), f"missing final encoder {src} (run s2i.shared --final first)"
        od = os.path.join(a.runs, v)
        os.makedirs(od, exist_ok=True)
        for s in subs:
            dst = os.path.join(od, f"enc_{s}.pt")
            if not os.path.exists(dst):
                shutil.copy(src, dst)
        log(f"{v}: encoder copied for {len(subs)} patients -> {od}")


# ----------------------------------------------------------------------------- report
def _load(run, name):
    out = {}
    for f in glob.glob(os.path.join(run, name, "readout_sub-P*.json")):
        d = json.load(open(f))
        out[d["subject"]] = d
    return out


def _ex(D, m, k="r"):
    return {s: r["explicit"][m]["mean"][k] for s, r in D.items() if m in r.get("explicit", {})
            and k in r["explicit"][m]["mean"]}


def compare(a_, b_, rng):
    from scipy.stats import wilcoxon
    s = sorted(set(a_) & set(b_))
    x, y = np.array([a_[q] for q in s]), np.array([b_[q] for q in s])
    d = x - y
    if len(d) == 0:
        return None
    bt = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(10000)])
    try:
        p = float(wilcoxon(x, y, alternative="two-sided").pvalue)
    except ValueError:
        p = float("nan")
    return {"n": len(d), "mean_a": float(x.mean()), "mean_b": float(y.mean()), "median_diff": float(np.median(d)),
            "mean_diff": float(d.mean()), "ci": [float(np.percentile(bt, 2.5)), float(np.percentile(bt, 97.5))],
            "wins": int((d > 0).sum()), "p": p, "per_patient": dict(zip(s, d.round(5).tolist()))}


def report(a):
    rng = np.random.default_rng(0)
    R = a.runs
    B = json.load(open(os.path.join(a.data, "p17_build.json")))["patients"]
    M = {}
    for lv in LEVELS:
        M[lv] = {"final": _load(os.path.join(R, "main"), f"final_{lv}"),
                 "learned3": _load(os.path.join(R, "no_env"), f"learned3_{lv}"),
                 "fixed3": _load(os.path.join(R, "no_env"), f"fixed3_{lv}")}
    fixed1 = _load(os.path.join(R, "no_env"), "fixed1_primary")
    f10 = _load(os.path.join(R, "main"), "final_primary_f10")
    P = M["primary"]
    included = sorted(s for s, i in B.items() if i["levels"].get("primary", {}).get("excluded") is None
                      and i["levels"].get("primary"))
    out, L = {}, ["# Phase 17: external validation on ScaleOLDNEW (protocol v1.6)", ""]
    have = {k: sorted(v) for k, v in P.items()}
    L.append(f"- included patients (primary level): {len(included)}: {', '.join(included)}")
    for k, v in have.items():
        if v != included:
            L.append(f"- **WARNING**: results for {k} exist for {len(v)} patients ({', '.join(v)}): run incomplete")
    # ---- confirmatory
    h1 = compare(_ex(P["final"], "combo_nosess"), _ex(P["final"], "base"), rng)
    h2 = compare(_ex(P["learned3"], "combo_nosess"), _ex(P["fixed3"], "combo_nosess"), rng)
    ps = [h1["p"], h2["p"]]
    order = np.argsort(ps)
    adj = [0.0, 0.0]
    adj[order[0]] = min(1.0, 2 * ps[order[0]])
    adj[order[1]] = max(adj[order[0]], min(1.0, ps[order[1]]))
    L += ["", "## Primary hypotheses (primary level, Pearson r; two-sided Wilcoxon, Holm over H1 and H2)", "",
          "| hypothesis | A | B | n | mean A | mean B | median A − B | mean A − B [95 % CI] | A > B | p | p (Holm) | supported |",
          "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for name, (la, lb), h, pa in (("H1 (system)", ("Final", "Base"), h1, adj[0]),
                                  ("H2 (learning)", ("Learned-3", "Fixed-3"), h2, adj[1])):
        sup = pa < 0.05 and h["median_diff"] > 0
        h["p_holm"], h["supported"] = pa, bool(sup)
        L.append(f"| {name} | {la} | {lb} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | {h['median_diff']:+.4f} | "
                 f"{h['mean_diff']:+.4f} [{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['wins']}/{h['n']} | {h['p']:.5f} | "
                 f"{pa:.5f} | {'**yes**' if sup else 'no'} |")
    out["H1"], out["H2"] = h1, h2
    L += ["", "Per-patient differences (A − B):", ""]
    for name, h in (("H1", h1), ("H2", h2)):
        L.append(f"- {name}: " + ", ".join(f"{s} {v:+.4f}" for s, v in h["per_patient"].items()))
    # ---- descriptive
    L += ["", "## Descriptive analyses (no confirmatory claim; unadjusted p)", "",
          "| level | comparison | metric | n | mean A | mean B | mean A − B [95 % CI] | median A − B | A > B | p |",
          "|---|---|---|---|---|---|---|---|---|---|"]
    desc = []

    def row(lv, lab, A, B, metric):
        h = compare(A, B, rng)
        if h is None:
            return
        desc.append({"level": lv, "comparison": lab, "metric": metric, **h})
        L.append(f"| {lv} | {lab} | {metric} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | {h['mean_diff']:+.4f} "
                 f"[{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['median_diff']:+.4f} | {h['wins']}/{h['n']} | {h['p']:.4f} |")
    row("primary", "Final − Fixed-1", _ex(P["final"], "combo_nosess"), _ex(fixed1, "combo_nosess"), "r")
    for lv in ("secondary", "sensitivity"):
        row(lv, "H1: Final − Base", _ex(M[lv]["final"], "combo_nosess"), _ex(M[lv]["final"], "base"), "r")
        row(lv, "H2: Learned-3 − Fixed-3", _ex(M[lv]["learned3"], "combo_nosess"), _ex(M[lv]["fixed3"], "combo_nosess"), "r")
    for metric in ("r_event", "r_no50", "r2"):
        row("primary", "Final − Base", _ex(P["final"], "combo_nosess", metric), _ex(P["final"], "base", metric), metric)
        row("primary", "Learned-3 − Fixed-3", _ex(P["learned3"], "combo_nosess", metric),
            _ex(P["fixed3"], "combo_nosess", metric), metric)
    mins = lambda D, f: np.mean([r["fracs"][f]["minutes"] for r in D.values()]) if D else float("nan")   # noqa: E731
    row("primary", f"Final, 10 % ({mins(f10, '0.1'):.1f} min) − Base, 20 % ({mins(P['final'], '0.2'):.1f} min)",
        _ex(f10, "combo_nosess"), _ex(P["final"], "base"), "r")
    out["descriptive"] = desc
    L += ["", "Mean over patients, primary level:", "", "| model | r | R² | r_event | r_no50 | constant predictions |",
          "|---|---|---|---|---|---|"]
    for lab, D, m in (("Base", P["final"], "base"), ("Fixed-1", fixed1, "combo_nosess"),
                      ("Fixed-3", P["fixed3"], "combo_nosess"), ("Learned-3", P["learned3"], "combo_nosess"),
                      ("Final", P["final"], "combo_nosess"), ("Final, 10 % calibration", f10, "combo_nosess")):
        v = {k: _ex(D, m, k) for k in ("r", "r2", "r_event", "r_no50")}
        nc = sum(r["explicit"][m].get("n_const", 0) for r in D.values() if m in r.get("explicit", {}))
        L.append(f"| {lab} | " + " | ".join(f"{np.mean(list(x.values())):.4f}" if x else "" for x in v.values())
                 + f" | {nc} |")
    # ---- common mode (Section 8)
    cm = {}
    for f in glob.glob(os.path.join(R, "review13", "volts_sub-P*.json")):
        d = json.load(open(f))
        cm[d["subject"]] = d
    if cm:
        L += ["", "## Reference / common-mode analysis (descriptive; lagged ridge, primary level, volts)", "",
              "| patient | setup | scalp-cm vs iEEG-cm r | mono, orig | mono − cm, orig | bip, orig | bip, CAR | cm, orig | "
              "removal of iEEG cm | interaction |", "|---|---|---|---|---|---|---|---|---|---|"]
        grp = {}
        for s in sorted(cm):
            q, p_ = cm[s], cm[s]["pred"]
            rem = p_["mono|orig"] - p_["mono_minus_cm|orig"]
            inter = p_["mono|orig"] - p_["mono|car"] - p_["bip|orig"] + p_["bip|car"]
            setup = B[s]["setup"]
            grp.setdefault(setup, []).append((q["r_scalp_cm_vs_ieeg_cm"], p_["mono|orig"], rem, inter))
            L.append(f"| {s} | {setup} | {q['r_scalp_cm_vs_ieeg_cm']:+.2f} | {p_['mono|orig']:.3f} | "
                     f"{p_['mono_minus_cm|orig']:.3f} | {p_['bip|orig']:.3f} | {p_['bip|car']:.3f} | {p_['cm|orig']:.3f} | "
                     f"{-rem:+.3f} | {inter:+.3f} |")
        L += ["", "| setup | n | mean scalp-cm vs iEEG-cm r | mean abs r | mean mono, orig | mean removal of iEEG cm | "
              "mean interaction |", "|---|---|---|---|---|---|---|"]
        for g, v in sorted(grp.items()):
            v = np.array(v)
            L.append(f"| {g} | {len(v)} | {v[:, 0].mean():+.3f} | {np.abs(v[:, 0]).mean():.3f} | {v[:, 1].mean():.3f} | "
                     f"{-v[:, 2].mean():+.3f} | {v[:, 3].mean():+.3f} |")
        out["common_mode"] = {s: {"setup": B[s]["setup"], **cm[s]} for s in cm}
    md = "\n".join(L)
    print(md)
    open(os.path.join(R, "p17_report.md"), "w", encoding="utf-8").write(md + "\n")
    json.dump(out, open(os.path.join(R, "p17_report.json"), "w"), indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=("build", "runs", "report"))
    ap.add_argument("--sc_root", help="ScaleOLDNEW root of Phase 14 (proc/, raw/)")
    ap.add_argument("--out", help="build: output root (one folder per level)")
    ap.add_argument("--data", help="runs: primary data folder; report: build output root")
    ap.add_argument("--enc_root", help="runs: folder with <variant>/enc_final.pt")
    ap.add_argument("--runs", help="runs / report: Phase-17 run root")
    ap.add_argument("--variants", nargs="*", default=["main", "seed1", "seed2", "short", "no_env"])
    ap.add_argument("--patients", nargs="*", default=None)
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    {"build": build, "runs": runs, "report": report}[a.cmd](a)


if __name__ == "__main__":
    main()

In [ ]:
#@title 2b. Kiểm tra (cả bốn số phải > 0)
!grep -c "def train_final" {CODE_DIR}/s2i/shared.py
!grep -c "def r_event" {CODE_DIR}/s2i/readout.py
!grep -c "PATTERN" {CODE_DIR}/s2i/review13.py
!grep -c "def build_patient" {CODE_DIR}/s2i/scales17.py

## 17A. Train 5 encoder cuối (protocol mục 3)

Mỗi biến thể dùng đúng siêu tham số và seed như trong bài:

| Biến thể | Cờ |
|---|---|
| main | seed 42 |
| seed1 | seed 1 |
| seed2 | seed 2 |
| short | `--half 40` (±0,2 s) |
| no_env | `--no_env` |

- 2 bệnh nhân early stopping được rút bằng seed của biến thể, giống cách chọn trong một fold LOSO; 13 bệnh nhân còn lại dùng để tối ưu.
- Quy tắc checkpoint: validate mỗi 150 bước, patience 6, tối đa 3000 bước; giữ checkpoint có val r tốt nhất.

Bước này **chưa đọc bất kỳ dữ liệu ScaleOLDNEW nào**.

In [ ]:
#@title 17A
VARIANTS = {'main': '--seed 42', 'seed1': '--seed 1', 'seed2': '--seed 2', 'short': '--half 40', 'no_env': '--no_env'}
for v, flags in VARIANTS.items():
    print('=' * 30, v, flags)
    !python -m s2i.shared --data {DATA_DIR} --out {ENC17}/{v} --final {flags} 2>&1 | grep -E "FINAL|final encoder|step .* \*|Error"
import torch
for v in VARIANTS:
    ck = torch.load(f'{ENC17}/{v}/enc_final.pt', weights_only=False, map_location='cpu')
    print(f"{v:7s} cfg={ck['cfg']} | val r {ck['hist'][0][1]:.4f} -> {ck['best_val']:.4f} | early stopping: {ck['val_subjects']}")

## 17B. Dựng dữ liệu ScaleOLDNEW + execution log (protocol mục 4, 5, 6, 9)

Bước này làm các việc sau (chưa chạy mô hình nào):
- Áp quy tắc kênh scalp và target.
- Đọc lại raw 4096 Hz để đếm giá trị non-finite và tìm khoảng mất tín hiệu (≥ 50 ms).
- Cắt cửa sổ 8 s, chia calib/gap/test, áp ngưỡng tiếp nhận (n_cal ≥ 5, ≥ 3 kênh scalp, ≥ 3 target hợp lệ).
- Ghi 3 mức đánh giá.

Log nằm ở `DATA17/p17_build.md`.

In [ ]:
#@title 17B
!python -m s2i.scales17 build --sc_root {SC_ROOT} --out {DATA17}

### ⏸ Dừng ở đây: gửi toàn bộ output 17B cho mình

Mình sẽ kiểm tra bốn điều (đều ở mức dữ liệu, chưa có kết quả mô hình):
1. Số kênh scalp giữ lại so với Table 1.
2. Số giá trị non-finite.
3. Số khoảng mất tín hiệu.
4. Số marker kích thích mỗi block (kỳ vọng 36 mỗi block: 12 mã `8`, 12 mã `16`, 12 mã `32`).

Nếu có lỗi **cài đặt** (code làm khác protocol), sửa trước khi chạy 17C. Mọi sai khác giữa log và protocol được ghi là deviation (mục 9).

## 17C. Readout ở mức chính (không QC tự động)

Các mô hình so sánh (protocol mục 3, bảng *Models compared*):

| Mô hình | Thành phần |
|---|---|
| Final | 5 encoder + LAG9 |
| Base | LAG9, có sẵn trong output của Final |
| Learned-3 | no_env + short + main, đã học |
| Fixed-3 | cùng ba kiến trúc ở trạng thái khởi tạo |
| Fixed-1 | no_env ở trạng thái khởi tạo |

Ngoài ra chạy Final với calib 10% trên đúng tập test, cho phân tích hiệu quả calib.

Mọi readout dùng `--strict`: chuẩn hoá chỉ bằng thống kê của cửa sổ calib, với cùng 5 fold calib.

In [ ]:
#@title 17C
P = f'{DATA17}/primary'
!python -m s2i.scales17 runs --enc_root {ENC17} --data {P} --runs {RUNS17}
R = RUNS17
!python -m s2i.readout --data {P} --run {R}/main --seed_runs {R}/seed1 {R}/seed2 {R}/short {R}/no_env --only_combo --strict --fracs 0.2 --explicit --out_name final_primary
!python -m s2i.readout --data {P} --run {R}/no_env --seed_runs {R}/short {R}/main --only_combo --strict --fracs 0.2 --explicit --out_name learned3_primary
!python -m s2i.readout --data {P} --run {R}/no_env --seed_runs {R}/short {R}/main --untrained --only_combo --strict --fracs 0.2 --explicit --out_name fixed3_primary
!python -m s2i.readout --data {P} --run {R}/no_env --untrained --only_combo --strict --fracs 0.2 --explicit --out_name fixed1_primary
!python -m s2i.readout --data {P} --run {R}/main --seed_runs {R}/seed1 {R}/seed2 {R}/short {R}/no_env --only_combo --strict --fracs 0.1 --explicit --explicit_frac 0.1 --out_name final_primary_f10

## 17D. H1 và H2 ở mức phụ (artefact-screened) và mức sensitivity (ngưỡng từ calib)

In [ ]:
#@title 17D
R = RUNS17
for LV in ('secondary', 'sensitivity'):
    D = f'{DATA17}/{LV}'
    !python -m s2i.readout --data {D} --run {R}/main --seed_runs {R}/seed1 {R}/seed2 {R}/short {R}/no_env --only_combo --strict --fracs 0.2 --explicit --out_name final_{LV}
    !python -m s2i.readout --data {D} --run {R}/no_env --seed_runs {R}/short {R}/main --only_combo --strict --fracs 0.2 --explicit --out_name learned3_{LV}
    !python -m s2i.readout --data {D} --run {R}/no_env --seed_runs {R}/short {R}/main --untrained --only_combo --strict --fracs 0.2 --explicit --out_name fixed3_{LV}

## 17E. Common mode theo cấu hình ampli (protocol mục 8, chỉ mô tả)

Lặp lại phân tích ở mục 3.9 của bài (tính bằng volt, rồi chuẩn hoá strict, dùng lagged ridge), ở mức chính.

In [ ]:
#@title 17E
!python -m s2i.review13 volts --data {DATA17}/primary --pattern "sub-P*.pt" --out {RUNS17}/review13

## 17F. Báo cáo

In [ ]:
#@title 17F
!python -m s2i.scales17 report --data {DATA17} --runs {RUNS17}

## Cách đọc kết quả
- **H1 và H2 là hai kết luận xác nhận duy nhất.**
  - Một giả thuyết được ủng hộ khi p sau hiệu chỉnh Holm < 0,05 **và** trung vị của hiệu theo bệnh nhân > 0.
  - Giả thuyết không được ủng hộ được báo cáo nổi bật ngang với giả thuyết được ủng hộ.
  - Không kết luận "tương đương" khi kết quả không có ý nghĩa thống kê.
- Mọi dòng còn lại chỉ là mô tả, p chưa hiệu chỉnh: mức phụ và mức sensitivity, r_event, r_no50, R², hiệu quả calib, Final − Fixed-1, common mode.
- `constant predictions` đếm số target mà mô hình dự đoán hằng số; theo quy ước của protocol, các target này được tính r = 0.

Gửi lại: output 17B (trước), sau đó toàn bộ output 17F.